# Temporal Phase Classifier — Complete Train, Evaluate and Export

This standalone Google Colab trains and evaluates the
**technique-conditioned ST-GCN/TCN temporal phase-classification
model** used by the Combat Cognition Framework.

Default input: the pinned Jab + Front Kick synthetic bootstrap bundle.
The bootstrap run validates the complete pipeline and produces
reproducible engineering evidence. **Synthetic scores are not
real-world martial-arts accuracy and must not be reported as such.**

The notebook produces:

- session-separated train/validation/test splits;
- three independently initialized training runs;
- accuracy, balanced accuracy, macro/weighted F1 and per-phase scores;
- a majority baseline and per-technique results;
- confusion matrix and phase-boundary measurements;
- coarse completed-sequence/repetition measurements;
- noise and missing-landmark robustness tests;
- PyTorch/ONNX parity and CPU latency;
- checkpoints, histories, hashes, provenance and a downloadable ZIP.


## 1. Environment

In Colab choose **Runtime → Change runtime type → T4 GPU**, then run
every cell in order.


In [ ]:
!pip -q install onnx onnxruntime scikit-learn pandas matplotlib seaborn


In [ ]:
import hashlib, importlib.util, json, os, platform, random, shutil, sys, time
import urllib.request
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Configuration and data acquisition

`DATA_SOURCE="github_sample"` makes the notebook runnable by anyone:
it downloads the exact bootstrap file from a pinned project commit.
Set `DATA_SOURCE="upload"` to select a compatible Temporal Data Lab
JSON export from the local computer.

The default dataset contains 24 synthetic Jab sessions and 24
synthetic Front Kick sessions. It is deliberately included because
the requested experiment is a bootstrap/pipeline evaluation.


In [ ]:
@dataclass
class Config:
    data_source: str = "github_sample"  # github_sample | upload
    window: int = 90
    stride: int = 15
    batch_size: int = 32
    epochs: int = 60
    patience: int = 10
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    hidden_size: int = 96
    dropout: float = 0.20
    seeds: tuple = (42, 43, 44)
    split_seed: int = 5299
    validation_fraction: float = 0.15
    test_fraction: float = 0.15
    boundary_tolerance_frames: int = 5
    boundary_smoothing_frames: int = 5

CFG = Config()
WORK_DIR = Path("/content/temporal_phase_research")
PIPELINE_DIR = WORK_DIR / "pipeline"
DATA_DIR = WORK_DIR / "data"
OUTPUT_ROOT = Path("/content/research_outputs/phase_classifier")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_ROOT / RUN_ID
for directory in (WORK_DIR, DATA_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

PINNED_COMMIT = "c1ae68b540b2bb4c9b70da7ce0c8783a5a9e0aae"
SAMPLE_URL = (
    "https://raw.githubusercontent.com/"
    "SachithBandaraThennakoon/martial-art-ai/"
    f"{PINNED_COMMIT}/training/temporal_phase/samples/"
    "universal-jab-front-kick-bootstrap.json"
)
DATA_FILE = DATA_DIR / "universal-jab-front-kick-bootstrap.json"

if CFG.data_source == "github_sample":
    print("Downloading pinned bootstrap data...")
    urllib.request.urlretrieve(SAMPLE_URL, DATA_FILE)
elif CFG.data_source == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one compatible JSON bundle.")
    name, payload = next(iter(uploaded.items()))
    DATA_FILE = DATA_DIR / name
    DATA_FILE.write_bytes(payload)
else:
    raise ValueError("DATA_SOURCE must be 'github_sample' or 'upload'.")

def sha256_file(path, block_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

print("Data:", DATA_FILE)
print("Bytes:", DATA_FILE.stat().st_size)
print("SHA-256:", sha256_file(DATA_FILE))


## 3. Embedded preparation and model implementation

This cell restores the exact preparation and model code into the
temporary Colab workspace. No Git clone or separate source-code
download is required.


In [ ]:
import base64, json, os, shutil
WORK_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR = WORK_DIR / "pipeline"
if PIPELINE_DIR.exists():
    shutil.rmtree(PIPELINE_DIR)
PIPELINE_DIR.mkdir(parents=True)
encoded_files = json.loads(r'''{"prepare_dataset.py":"IiIiQnVpbGQgYSBsZWFrYWdlLXNhZmUgdGVtcG9yYWwgcGhhc2UgZGF0YXNldCBmcm9tIGV4cG9ydGVkIFByYWN0aWNlIHRhcGVzLgoKVXNhZ2UgaW4gQ29sYWIgb3IgbG9jYWxseToKICBweXRob24gcHJlcGFyZV9kYXRhc2V0LnB5IFwKICAgIC0taW5wdXQtZGlyIC9jb250ZW50L3RhcGVzIFwKICAgIC0tb3V0cHV0IC9jb250ZW50L2phYl90ZW1wb3JhbF9kYXRhc2V0Lm5weiBcCiAgICAtLXRlY2huaXF1ZSBqYWIgXAogICAgLS1zdGF0ZXMgLi4vLi4vYmFja2VuZC9kYXRhL3RlY2huaXF1ZXMvamFiL3N0YXRlcy5qc29uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKClBBRCA9ICJfX1BBRF9fIgpVTktOT1dOID0gIl9fVU5LTk9XTl9fIgpUUkFDS0lOR19MT1NUID0gIl9fVFJBQ0tJTkdfTE9TVF9fIgpKT0lOVF9DT1VOVCA9IDMzCkNIQU5ORUxfQ09VTlQgPSA0CgoKZGVmIHBhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWlucHV0LWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10ZWNobmlxdWUiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zdGF0ZXMiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlcXVlbmNlLWxlbmd0aCIsIHR5cGU9aW50LCBkZWZhdWx0PTkwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zdHJpZGUiLCB0eXBlPWludCwgZGVmYXVsdD0xNSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWluaW11bS1sYWJlbGxlZC1yYXRpbyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC42NSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0taW5jbHVkZS1zeW50aGV0aWMiLAogICAgICAgIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgaGVscD0iSW5jbHVkZSBleHBsaWNpdGx5IG1hcmtlZCBib290c3RyYXAgc3ludGhldGljIHNlc3Npb25zLiIsCiAgICApCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCmRlZiBsb2FkX3N0YXRlX25hbWVzKHBhdGg6IFBhdGgpIC0+IGxpc3Rbc3RyXToKICAgIGRvY3VtZW50ID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIG9yZGVyID0gZG9jdW1lbnQuZ2V0KCJzdGF0ZV9vcmRlciIpIG9yIGxpc3QoZG9jdW1lbnQuZ2V0KCJzdGF0ZXMiLCB7fSkpCiAgICBpZiBub3Qgb3JkZXI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk5vIHN0YXRlIGRlZmluaXRpb25zIGZvdW5kIGluIHtwYXRofSIpCiAgICByZXR1cm4gW1BBRCwgVU5LTk9XTiwgVFJBQ0tJTkdfTE9TVCwgKm9yZGVyXQoKCmRlZiByZXN0b3JlX2xhbmRtYXJrcyh2YWx1ZXM6IEFueSkgLT4gbnAubmRhcnJheToKICAgIHJlc3VsdCA9IG5wLnplcm9zKChKT0lOVF9DT1VOVCwgQ0hBTk5FTF9DT1VOVCksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZXMsIGxpc3QpOgogICAgICAgIHJldHVybiByZXN1bHQKICAgIGZvciBpbmRleCwgcG9pbnQgaW4gZW51bWVyYXRlKHZhbHVlc1s6Sk9JTlRfQ09VTlRdKToKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwb2ludCwgbGlzdCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIGNoYW5uZWwgaW4gcmFuZ2UobWluKENIQU5ORUxfQ09VTlQsIGxlbihwb2ludCkpKToKICAgICAgICAgICAgdmFsdWUgPSBwb2ludFtjaGFubmVsXQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoaW50LCBmbG9hdCkpOgogICAgICAgICAgICAgICAgcmVzdWx0W2luZGV4LCBjaGFubmVsXSA9IGZsb2F0KHZhbHVlKSAvIDEwMDAwLjAKICAgIHJldHVybiByZXN1bHQKCgpkZWYgc2VsZWN0X2xhbmRtYXJrcyhmcmFtZTogZGljdFtzdHIsIEFueV0pIC0+IG5wLm5kYXJyYXk6CiAgICBmb3Iga2V5IGluICgid3AiLCAib3AiLCAicCIpOgogICAgICAgIHZhbHVlcyA9IGZyYW1lLmdldChrZXkpCiAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZXMsIGxpc3QpIGFuZCB2YWx1ZXM6CiAgICAgICAgICAgIHJldHVybiByZXN0b3JlX2xhbmRtYXJrcyh2YWx1ZXMpCiAgICByZXR1cm4gbnAuemVyb3MoKEpPSU5UX0NPVU5ULCBDSEFOTkVMX0NPVU5UKSwgZHR5cGU9bnAuZmxvYXQzMikKCgpkZWYgbm9ybWFsaXplX3Bvc2UocG9zZTogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgIG5vcm1hbGl6ZWQgPSBwb3NlLmNvcHkoKQogICAgbGVmdF9oaXAsIHJpZ2h0X2hpcCA9IG5vcm1hbGl6ZWRbMjMsIDozXSwgbm9ybWFsaXplZFsyNCwgOjNdCiAgICBsZWZ0X3Nob3VsZGVyLCByaWdodF9zaG91bGRlciA9IG5vcm1hbGl6ZWRbMTEsIDozXSwgbm9ybWFsaXplZFsxMiwgOjNdCiAgICByb290ID0gKGxlZnRfaGlwICsgcmlnaHRfaGlwKSAvIDIuMAogICAgc2hvdWxkZXJfd2lkdGggPSBucC5saW5hbGcubm9ybShsZWZ0X3Nob3VsZGVyIC0gcmlnaHRfc2hvdWxkZXIpCiAgICB0b3Jzb19oZWlnaHQgPSBucC5saW5hbGcubm9ybSgKICAgICAgICAoKGxlZnRfc2hvdWxkZXIgKyByaWdodF9zaG91bGRlcikgLyAyLjApIC0gcm9vdAogICAgKQogICAgc2NhbGUgPSBtYXgoZmxvYXQoc2hvdWxkZXJfd2lkdGgpLCBmbG9hdCh0b3Jzb19oZWlnaHQpLCAxZS00KQogICAgbm9ybWFsaXplZFs6LCA6M10gPSAobm9ybWFsaXplZFs6LCA6M10gLSByb290KSAvIHNjYWxlCiAgICBub3JtYWxpemVkWzosIDNdID0gbnAuY2xpcChub3JtYWxpemVkWzosIDNdLCAwLjAsIDEuMCkKICAgIHJldHVybiBub3JtYWxpemVkCgoKZGVmIGNvcnJlY3RlZF9sYWJlbChmcmFtZTogZGljdFtzdHIsIEFueV0pIC0+IHN0cjoKICAgIGNvcnJlY3RlZCA9IGZyYW1lLmdldCgicmMiKSBvciB7fQogICAgaWYgY29ycmVjdGVkLmdldCgidGwiKSBpcyBUcnVlOgogICAgICAgIHJldHVybiBUUkFDS0lOR19MT1NUCiAgICBpZiBjb3JyZWN0ZWQuZ2V0KCJ1IikgaXMgVHJ1ZToKICAgICAgICByZXR1cm4gVU5LTk9XTgogICAgc3RhdGUgPSBjb3JyZWN0ZWQuZ2V0KCJzIikKICAgIHJldHVybiBzdHIoc3RhdGUpIGlmIHN0YXRlIGVsc2UgVU5LTk9XTgoKCmRlZiB2ZXJpZmllZF9tYW51YWxfbGFiZWxzKAogICAgZG9jdW1lbnQ6IGRpY3Rbc3RyLCBBbnldLCBmcmFtZV9jb3VudDogaW50LCBpbmNsdWRlX3N5bnRoZXRpYzogYm9vbAopIC0+IGxpc3Rbc3RyXSB8IE5vbmU6CiAgICBhbm5vdGF0aW9uID0gZG9jdW1lbnQuZ2V0KCJtYW51YWxfYW5ub3RhdGlvbiIpIG9yIHt9CiAgICBhY2NlcHRlZCA9IHsiaHVtYW5fdmVyaWZpZWQifQogICAgaWYgaW5jbHVkZV9zeW50aGV0aWM6CiAgICAgICAgYWNjZXB0ZWQuYWRkKCJzeW50aGV0aWNfdmVyaWZpZWQiKQogICAgaWYgYW5ub3RhdGlvbi5nZXQoInN0YXR1cyIpIG5vdCBpbiBhY2NlcHRlZDoKICAgICAgICByZXR1cm4gTm9uZQogICAgbGFiZWxzID0gW1BBRF0gKiBmcmFtZV9jb3VudAogICAgZm9yIHNlZ21lbnQgaW4gYW5ub3RhdGlvbi5nZXQoInNlZ21lbnRzIikgb3IgW106CiAgICAgICAgc3RhcnQgPSBpbnQoc2VnbWVudC5nZXQoInN0YXJ0X2ZyYW1lIiwgLTEpKQogICAgICAgIGVuZCA9IGludChzZWdtZW50LmdldCgiZW5kX2ZyYW1lIiwgLTEpKQogICAgICAgIHN0YXRlID0gc3RyKHNlZ21lbnQuZ2V0KCJzdGF0ZSIpIG9yICIiKS5zdHJpcCgpLnVwcGVyKCkKICAgICAgICBpZiBzdGFydCA8IDAgb3IgZW5kIDwgc3RhcnQgb3IgZW5kID49IGZyYW1lX2NvdW50IG9yIG5vdCBzdGF0ZToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2Uoc3RhcnQsIGVuZCArIDEpOgogICAgICAgICAgICBpZiBsYWJlbHNbaW5kZXhdICE9IFBBRDoKICAgICAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgICAgIGxhYmVsc1tpbmRleF0gPSBzdGF0ZQogICAgcmV0dXJuIGxhYmVscyBpZiBsYWJlbHMgYW5kIGFsbChsYWJlbCAhPSBQQUQgZm9yIGxhYmVsIGluIGxhYmVscykgZWxzZSBOb25lCgoKZGVmIHRhcGVfbWF0Y2hlc190ZWNobmlxdWUoZG9jdW1lbnQ6IGRpY3Rbc3RyLCBBbnldLCB0ZWNobmlxdWU6IHN0cikgLT4gYm9vbDoKICAgIG1ldGFkYXRhID0gZG9jdW1lbnQuZ2V0KCJtZXRhZGF0YSIpIG9yIHt9CiAgICBuYW1lID0gKAogICAgICAgIGRvY3VtZW50LmdldCgidGVjaG5pcXVlX2lkIikKICAgICAgICBvciBkb2N1bWVudC5nZXQoInRlY2huaXF1ZV9uYW1lIikKICAgICAgICBvciBtZXRhZGF0YS5nZXQoInRlY2huaXF1ZUlkIikKICAgICAgICBvciBtZXRhZGF0YS5nZXQoInRlY2huaXF1ZV9pZCIpCiAgICAgICAgb3IgbWV0YWRhdGEuZ2V0KCJ0ZWNobmlxdWVOYW1lIikKICAgICAgICBvciBtZXRhZGF0YS5nZXQoInRlY2huaXF1ZV9uYW1lIikKICAgICAgICBvciAiIgogICAgKQogICAgbm9ybWFsaXplZCA9IHN0cihuYW1lKS5zdHJpcCgpLmxvd2VyKCkucmVwbGFjZSgiICIsICItIikKICAgIHJldHVybiBub3Qgbm9ybWFsaXplZCBvciBub3JtYWxpemVkID09IHRlY2huaXF1ZS5sb3dlcigpCgoKZGVmIHdpbmRvd3NfZm9yX3Nlc3Npb24oCiAgICBkb2N1bWVudDogZGljdFtzdHIsIEFueV0sCiAgICBsYWJlbF90b19pZDogZGljdFtzdHIsIGludF0sCiAgICBzZXF1ZW5jZV9sZW5ndGg6IGludCwKICAgIHN0cmlkZTogaW50LAogICAgbWluaW11bV9sYWJlbGxlZF9yYXRpbzogZmxvYXQsCiAgICBpbmNsdWRlX3N5bnRoZXRpYzogYm9vbCwKKSAtPiB0dXBsZVtsaXN0W25wLm5kYXJyYXldLCBsaXN0W25wLm5kYXJyYXldLCBsaXN0W25wLm5kYXJyYXldXToKICAgIGZyYW1lcyA9IGRvY3VtZW50LmdldCgiZnJhbWVzIikgb3IgW10KICAgIG1hbnVhbF9sYWJlbHMgPSB2ZXJpZmllZF9tYW51YWxfbGFiZWxzKGRvY3VtZW50LCBsZW4oZnJhbWVzKSwgaW5jbHVkZV9zeW50aGV0aWMpCiAgICBpZiBtYW51YWxfbGFiZWxzIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFtdLCBbXSwgW10KICAgIGZlYXR1cmVzID0gbnAuYXNhcnJheSgKICAgICAgICBbbm9ybWFsaXplX3Bvc2Uoc2VsZWN0X2xhbmRtYXJrcyhmcmFtZSkpIGZvciBmcmFtZSBpbiBmcmFtZXNdLAogICAgICAgIGR0eXBlPW5wLmZsb2F0MzIsCiAgICApCiAgICBsYWJlbHMgPSBucC5hc2FycmF5KAogICAgICAgIFtsYWJlbF90b19pZC5nZXQobGFiZWwsIGxhYmVsX3RvX2lkW1VOS05PV05dKSBmb3IgbGFiZWwgaW4gbWFudWFsX2xhYmVsc10sCiAgICAgICAgZHR5cGU9bnAuaW50NjQsCiAgICApCiAgICBsYWJlbGxlZCA9IG5wLm9uZXMobGVuKGZyYW1lcyksIGR0eXBlPW5wLmJvb2xfKQogICAgb3V0cHV0czogbGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICB0YXJnZXRzOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgIG1hc2tzOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgIGlmIG5vdCBsZW4oZmVhdHVyZXMpOgogICAgICAgIHJldHVybiBvdXRwdXRzLCB0YXJnZXRzLCBtYXNrcwoKICAgIHN0YXJ0cyA9IGxpc3QocmFuZ2UoMCwgbWF4KDEsIGxlbihmZWF0dXJlcykgLSBzZXF1ZW5jZV9sZW5ndGggKyAxKSwgc3RyaWRlKSkKICAgIGxhc3Rfc3RhcnQgPSBtYXgoMCwgbGVuKGZlYXR1cmVzKSAtIHNlcXVlbmNlX2xlbmd0aCkKICAgIGlmIGxhc3Rfc3RhcnQgbm90IGluIHN0YXJ0czoKICAgICAgICBzdGFydHMuYXBwZW5kKGxhc3Rfc3RhcnQpCgogICAgZm9yIHN0YXJ0IGluIHN0YXJ0czoKICAgICAgICBlbmQgPSBtaW4obGVuKGZlYXR1cmVzKSwgc3RhcnQgKyBzZXF1ZW5jZV9sZW5ndGgpCiAgICAgICAgdmFsaWRfY291bnQgPSBlbmQgLSBzdGFydAogICAgICAgIGlmIHZhbGlkX2NvdW50IDw9IDA6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgZmxvYXQobGFiZWxsZWRbc3RhcnQ6ZW5kXS5tZWFuKCkpIDwgbWluaW11bV9sYWJlbGxlZF9yYXRpbzoKICAgICAgICAgICAgY29udGludWUKICAgICAgICB4ID0gbnAuemVyb3MoCiAgICAgICAgICAgIChzZXF1ZW5jZV9sZW5ndGgsIEpPSU5UX0NPVU5ULCBDSEFOTkVMX0NPVU5UKSwgZHR5cGU9bnAuZmxvYXQzMgogICAgICAgICkKICAgICAgICB5ID0gbnAuZnVsbChzZXF1ZW5jZV9sZW5ndGgsIGxhYmVsX3RvX2lkW1BBRF0sIGR0eXBlPW5wLmludDY0KQogICAgICAgIG1hc2sgPSBucC56ZXJvcyhzZXF1ZW5jZV9sZW5ndGgsIGR0eXBlPW5wLmJvb2xfKQogICAgICAgIHhbOnZhbGlkX2NvdW50XSA9IGZlYXR1cmVzW3N0YXJ0OmVuZF0KICAgICAgICB5Wzp2YWxpZF9jb3VudF0gPSBsYWJlbHNbc3RhcnQ6ZW5kXQogICAgICAgIG1hc2tbOnZhbGlkX2NvdW50XSA9IFRydWUKICAgICAgICBvdXRwdXRzLmFwcGVuZCh4KQogICAgICAgIHRhcmdldHMuYXBwZW5kKHkpCiAgICAgICAgbWFza3MuYXBwZW5kKG1hc2spCiAgICByZXR1cm4gb3V0cHV0cywgdGFyZ2V0cywgbWFza3MKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcmdzID0gcGFyc2VfYXJncygpCiAgICBpZiBhcmdzLnNlcXVlbmNlX2xlbmd0aCA8IDggb3IgYXJncy5zdHJpZGUgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNlcXVlbmNlIGxlbmd0aCBtdXN0IGJlID49IDggYW5kIHN0cmlkZSBtdXN0IGJlID49IDEiKQogICAgbGFiZWxfbmFtZXMgPSBsb2FkX3N0YXRlX25hbWVzKGFyZ3Muc3RhdGVzKQogICAgbGFiZWxfdG9faWQgPSB7bmFtZTogaW5kZXggZm9yIGluZGV4LCBuYW1lIGluIGVudW1lcmF0ZShsYWJlbF9uYW1lcyl9CiAgICBmZWF0dXJlX3dpbmRvd3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgbGFiZWxfd2luZG93czogbGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICBtYXNrX3dpbmRvd3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgZ3JvdXBzOiBsaXN0W3N0cl0gPSBbXQogICAgb3JpZ2luczogbGlzdFtzdHJdID0gW10KCiAgICBwYXRocyA9IHNvcnRlZChhcmdzLmlucHV0X2Rpci5nbG9iKCIqLmpzb24iKSkKICAgIGZvciBwYXRoIGluIHBhdGhzOgogICAgICAgIHNvdXJjZSA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgZG9jdW1lbnRzID0gc291cmNlLmdldCgic2Vzc2lvbnMiKSBpZiBpc2luc3RhbmNlKHNvdXJjZSwgZGljdCkgZWxzZSBOb25lCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoZG9jdW1lbnRzLCBsaXN0KToKICAgICAgICAgICAgZG9jdW1lbnRzID0gW3NvdXJjZV0KICAgICAgICBmb3IgZG9jdW1lbnRfaW5kZXgsIGRvY3VtZW50IGluIGVudW1lcmF0ZShkb2N1bWVudHMpOgogICAgICAgICAgICBpZiBub3QgdGFwZV9tYXRjaGVzX3RlY2huaXF1ZShkb2N1bWVudCwgYXJncy50ZWNobmlxdWUpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgeCwgeSwgbWFza3MgPSB3aW5kb3dzX2Zvcl9zZXNzaW9uKAogICAgICAgICAgICAgICAgZG9jdW1lbnQsCiAgICAgICAgICAgICAgICBsYWJlbF90b19pZCwKICAgICAgICAgICAgICAgIGFyZ3Muc2VxdWVuY2VfbGVuZ3RoLAogICAgICAgICAgICAgICAgYXJncy5zdHJpZGUsCiAgICAgICAgICAgICAgICBhcmdzLm1pbmltdW1fbGFiZWxsZWRfcmF0aW8sCiAgICAgICAgICAgICAgICBhcmdzLmluY2x1ZGVfc3ludGhldGljLAogICAgICAgICAgICApCiAgICAgICAgICAgIGdyb3VwID0gc3RyKAogICAgICAgICAgICAgICAgZG9jdW1lbnQuZ2V0KCJzZXNzaW9uX2lkIikKICAgICAgICAgICAgICAgIG9yIChkb2N1bWVudC5nZXQoIm1ldGFkYXRhIikgb3Ige30pLmdldCgic2Vzc2lvbklkIikKICAgICAgICAgICAgICAgIG9yIGYie3BhdGguc3RlbX1fe2RvY3VtZW50X2luZGV4fSIKICAgICAgICAgICAgKQogICAgICAgICAgICBmZWF0dXJlX3dpbmRvd3MuZXh0ZW5kKHgpCiAgICAgICAgICAgIGxhYmVsX3dpbmRvd3MuZXh0ZW5kKHkpCiAgICAgICAgICAgIG1hc2tfd2luZG93cy5leHRlbmQobWFza3MpCiAgICAgICAgICAgIGdyb3Vwcy5leHRlbmQoW2dyb3VwXSAqIGxlbih4KSkKICAgICAgICAgICAgb3JpZ2luID0gc3RyKAogICAgICAgICAgICAgICAgKGRvY3VtZW50LmdldCgicHJvdmVuYW5jZSIpIG9yIHt9KS5nZXQoIm9yaWdpbiIpIG9yICJyZWFsIgogICAgICAgICAgICApCiAgICAgICAgICAgIG9yaWdpbnMuZXh0ZW5kKFtvcmlnaW5dICogbGVuKHgpKQoKICAgIGlmIG5vdCBmZWF0dXJlX3dpbmRvd3M6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAiTm8gaHVtYW4tdmVyaWZpZWQgd2luZG93cyB3ZXJlIGNyZWF0ZWQuIENvbXBsZXRlIG1hbnVhbCBhbm5vdGF0aW9uICIKICAgICAgICAgICAgImFuZCBleHBvcnQgdmVyaWZpZWQgdGFwZSBidW5kbGVzIGZyb20gdGhlIFRlbXBvcmFsIERhdGEgTGFiLiIKICAgICAgICApCgogICAgYXJncy5vdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoCiAgICAgICAgYXJncy5vdXRwdXQsCiAgICAgICAgZmVhdHVyZXM9bnAuc3RhY2soZmVhdHVyZV93aW5kb3dzKSwKICAgICAgICBsYWJlbHM9bnAuc3RhY2sobGFiZWxfd2luZG93cyksCiAgICAgICAgbWFzaz1ucC5zdGFjayhtYXNrX3dpbmRvd3MpLAogICAgICAgIGdyb3Vwcz1ucC5hc2FycmF5KGdyb3VwcyksCiAgICAgICAgb3JpZ2lucz1ucC5hc2FycmF5KG9yaWdpbnMpLAogICAgICAgIGxhYmVsX25hbWVzPW5wLmFzYXJyYXkobGFiZWxfbmFtZXMpLAogICAgICAgIHRlY2huaXF1ZT1ucC5hc2FycmF5KGFyZ3MudGVjaG5pcXVlKSwKICAgICAgICBzZXF1ZW5jZV9sZW5ndGg9bnAuYXNhcnJheShhcmdzLnNlcXVlbmNlX2xlbmd0aCksCiAgICAgICAgc2NoZW1hX3ZlcnNpb249bnAuYXNhcnJheSgiMS4wIiksCiAgICApCiAgICBwcmludChmIldyb3RlIHtsZW4oZmVhdHVyZV93aW5kb3dzKX0gd2luZG93cyBmcm9tIHtsZW4oc2V0KGdyb3VwcykpfSBzZXNzaW9ucyIpCiAgICBwcmludChmIk91dHB1dDoge2FyZ3Mub3V0cHV0fSIpCiAgICBwcmludChmIkxhYmVsczoge2xhYmVsX25hbWVzfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=","prepare_universal_dataset.py":"IiIiQnVpbGQgb25lIHRlY2huaXF1ZS1jb25kaXRpb25lZCB0ZW1wb3JhbCBkYXRhc2V0IGZyb20gdmVyaWZpZWQgZXhwb3J0cy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY29weQppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBwcmVwYXJlX2RhdGFzZXQgaW1wb3J0IHdpbmRvd3NfZm9yX3Nlc3Npb24KCgpkZWYgcGFyc2VfYXJncygpIC0+IGFyZ3BhcnNlLk5hbWVzcGFjZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taW5wdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLWxhYmVsLWNvbmZpZyIsCiAgICAgICAgdHlwZT1QYXRoLAogICAgICAgIGRlZmF1bHQ9UGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJ1bml2ZXJzYWwtbGFiZWxzLmpzb24iKSwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VxdWVuY2UtbGVuZ3RoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OTApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXN0cmlkZSIsIHR5cGU9aW50LCBkZWZhdWx0PTE1KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1taW5pbXVtLWxhYmVsbGVkLXJhdGlvIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjY1KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1pbmNsdWRlLXN5bnRoZXRpYyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCmRlZiB0ZWNobmlxdWVfaWQoZG9jdW1lbnQ6IGRpY3QpIC0+IHN0cjoKICAgIG1ldGFkYXRhID0gZG9jdW1lbnQuZ2V0KCJtZXRhZGF0YSIpIG9yIHt9CiAgICB2YWx1ZSA9ICgKICAgICAgICBkb2N1bWVudC5nZXQoInRlY2huaXF1ZV9pZCIpCiAgICAgICAgb3IgZG9jdW1lbnQuZ2V0KCJ0ZWNobmlxdWVfbmFtZSIpCiAgICAgICAgb3IgbWV0YWRhdGEuZ2V0KCJ0ZWNobmlxdWVJZCIpCiAgICAgICAgb3IgbWV0YWRhdGEuZ2V0KCJ0ZWNobmlxdWVfaWQiKQogICAgICAgIG9yIG1ldGFkYXRhLmdldCgidGVjaG5pcXVlTmFtZSIpCiAgICAgICAgb3IgbWV0YWRhdGEuZ2V0KCJ0ZWNobmlxdWVfbmFtZSIpCiAgICAgICAgb3IgIiIKICAgICkKICAgIHJldHVybiBzdHIodmFsdWUpLnN0cmlwKCkubG93ZXIoKS5yZXBsYWNlKCIgIiwgIi0iKQoKCmRlZiByZW1hcF9hbm5vdGF0aW9uKGRvY3VtZW50OiBkaWN0LCBtYXBwaW5nOiBkaWN0W3N0ciwgc3RyXSkgLT4gZGljdDoKICAgIHJlbWFwcGVkID0gY29weS5kZWVwY29weShkb2N1bWVudCkKICAgIGFubm90YXRpb24gPSByZW1hcHBlZC5nZXQoIm1hbnVhbF9hbm5vdGF0aW9uIikgb3Ige30KICAgIGZvciBzZWdtZW50IGluIGFubm90YXRpb24uZ2V0KCJzZWdtZW50cyIpIG9yIFtdOgogICAgICAgIG5hdGl2ZSA9IHN0cihzZWdtZW50LmdldCgic3RhdGUiKSBvciAiIikuc3RyaXAoKS51cHBlcigpCiAgICAgICAgaWYgbmF0aXZlIGluIHsiX19VTktOT1dOX18iLCAiX19UUkFDS0lOR19MT1NUX18ifToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBuYXRpdmUgbm90IGluIG1hcHBpbmc6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidObyB1bml2ZXJzYWwgcGhhc2UgbWFwcGluZyBmb3Igc3RhdGUgIntuYXRpdmV9IicpCiAgICAgICAgc2VnbWVudFsic3RhdGUiXSA9IG1hcHBpbmdbbmF0aXZlXQogICAgcmV0dXJuIHJlbWFwcGVkCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQogICAgY29uZmlnID0ganNvbi5sb2FkcyhhcmdzLmxhYmVsX2NvbmZpZy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBwaGFzZXMgPSBjb25maWdbInBoYXNlcyJdCiAgICBsYWJlbF90b19pZCA9IHtuYW1lOiBpbmRleCBmb3IgaW5kZXgsIG5hbWUgaW4gZW51bWVyYXRlKHBoYXNlcyl9CiAgICBzdXBwb3J0ZWQgPSBjb25maWdbInRlY2huaXF1ZXMiXQogICAgdGVjaG5pcXVlX25hbWVzID0gbGlzdChzdXBwb3J0ZWQpCiAgICB0ZWNobmlxdWVfdG9faWQgPSB7bmFtZTogaW5kZXggZm9yIGluZGV4LCBuYW1lIGluIGVudW1lcmF0ZSh0ZWNobmlxdWVfbmFtZXMpfQoKICAgIGZlYXR1cmVzLCBsYWJlbHMsIG1hc2tzID0gW10sIFtdLCBbXQogICAgZ3JvdXBzLCBvcmlnaW5zLCB0ZWNobmlxdWVfaWRzID0gW10sIFtdLCBbXQogICAgZm9yIHBhdGggaW4gc29ydGVkKGFyZ3MuaW5wdXRfZGlyLnJnbG9iKCIqLmpzb24iKSk6CiAgICAgICAgc291cmNlID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBkb2N1bWVudHMgPSBzb3VyY2UuZ2V0KCJzZXNzaW9ucyIpIGlmIGlzaW5zdGFuY2Uoc291cmNlLCBkaWN0KSBlbHNlIE5vbmUKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShkb2N1bWVudHMsIGxpc3QpOgogICAgICAgICAgICBkb2N1bWVudHMgPSBbc291cmNlXQogICAgICAgIGZvciBkb2N1bWVudF9pbmRleCwgZG9jdW1lbnQgaW4gZW51bWVyYXRlKGRvY3VtZW50cyk6CiAgICAgICAgICAgIHRlY2huaXF1ZSA9IHRlY2huaXF1ZV9pZChkb2N1bWVudCkKICAgICAgICAgICAgaWYgdGVjaG5pcXVlIG5vdCBpbiBzdXBwb3J0ZWQ6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByZW1hcHBlZCA9IHJlbWFwX2Fubm90YXRpb24oCiAgICAgICAgICAgICAgICBkb2N1bWVudCwgc3VwcG9ydGVkW3RlY2huaXF1ZV1bIm5hdGl2ZV90b19waGFzZSJdCiAgICAgICAgICAgICkKICAgICAgICAgICAgeCwgeSwgdmFsaWQgPSB3aW5kb3dzX2Zvcl9zZXNzaW9uKAogICAgICAgICAgICAgICAgcmVtYXBwZWQsCiAgICAgICAgICAgICAgICBsYWJlbF90b19pZCwKICAgICAgICAgICAgICAgIGFyZ3Muc2VxdWVuY2VfbGVuZ3RoLAogICAgICAgICAgICAgICAgYXJncy5zdHJpZGUsCiAgICAgICAgICAgICAgICBhcmdzLm1pbmltdW1fbGFiZWxsZWRfcmF0aW8sCiAgICAgICAgICAgICAgICBhcmdzLmluY2x1ZGVfc3ludGhldGljLAogICAgICAgICAgICApCiAgICAgICAgICAgIHNlc3Npb24gPSBzdHIoCiAgICAgICAgICAgICAgICBkb2N1bWVudC5nZXQoInNlc3Npb25faWQiKQogICAgICAgICAgICAgICAgb3IgKGRvY3VtZW50LmdldCgibWV0YWRhdGEiKSBvciB7fSkuZ2V0KCJzZXNzaW9uSWQiKQogICAgICAgICAgICAgICAgb3IgZiJ7cGF0aC5zdGVtfV97ZG9jdW1lbnRfaW5kZXh9IgogICAgICAgICAgICApCiAgICAgICAgICAgIG9yaWdpbiA9IHN0cigKICAgICAgICAgICAgICAgIChkb2N1bWVudC5nZXQoInByb3ZlbmFuY2UiKSBvciB7fSkuZ2V0KCJvcmlnaW4iKSBvciAicmVhbCIKICAgICAgICAgICAgKQogICAgICAgICAgICBmZWF0dXJlcy5leHRlbmQoeCkKICAgICAgICAgICAgbGFiZWxzLmV4dGVuZCh5KQogICAgICAgICAgICBtYXNrcy5leHRlbmQodmFsaWQpCiAgICAgICAgICAgIGdyb3Vwcy5leHRlbmQoW2Yie3RlY2huaXF1ZX06e3Nlc3Npb259Il0gKiBsZW4oeCkpCiAgICAgICAgICAgIG9yaWdpbnMuZXh0ZW5kKFtvcmlnaW5dICogbGVuKHgpKQogICAgICAgICAgICB0ZWNobmlxdWVfaWRzLmV4dGVuZChbdGVjaG5pcXVlX3RvX2lkW3RlY2huaXF1ZV1dICogbGVuKHgpKQoKICAgIGlmIG5vdCBmZWF0dXJlczoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIHZlcmlmaWVkIG11bHRpLXRlY2huaXF1ZSB3aW5kb3dzIHdlcmUgY3JlYXRlZCIpCiAgICBvYnNlcnZlZF90ZWNobmlxdWVzID0gewogICAgICAgIHRlY2huaXF1ZV9uYW1lc1tpbmRleF0gZm9yIGluZGV4IGluIHNldCh0ZWNobmlxdWVfaWRzKQogICAgfQogICAgaWYgbGVuKG9ic2VydmVkX3RlY2huaXF1ZXMpIDwgMjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICJVbml2ZXJzYWwgdHJhaW5pbmcgcmVxdWlyZXMgdmVyaWZpZWQgc2Vzc2lvbnMgZnJvbSBhdCBsZWFzdCB0d28gIgogICAgICAgICAgICAidGVjaG5pcXVlczsgb2JzZXJ2ZWQ6ICIKICAgICAgICAgICAgKyAiLCAiLmpvaW4oc29ydGVkKG9ic2VydmVkX3RlY2huaXF1ZXMpKQogICAgICAgICkKICAgIGFyZ3Mub3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBucC5zYXZlel9jb21wcmVzc2VkKAogICAgICAgIGFyZ3Mub3V0cHV0LAogICAgICAgIGZlYXR1cmVzPW5wLnN0YWNrKGZlYXR1cmVzKSwKICAgICAgICBsYWJlbHM9bnAuc3RhY2sobGFiZWxzKSwKICAgICAgICBtYXNrPW5wLnN0YWNrKG1hc2tzKSwKICAgICAgICBncm91cHM9bnAuYXNhcnJheShncm91cHMpLAogICAgICAgIG9yaWdpbnM9bnAuYXNhcnJheShvcmlnaW5zKSwKICAgICAgICB0ZWNobmlxdWVfaWRzPW5wLmFzYXJyYXkodGVjaG5pcXVlX2lkcywgZHR5cGU9bnAuaW50NjQpLAogICAgICAgIHRlY2huaXF1ZV9uYW1lcz1ucC5hc2FycmF5KHRlY2huaXF1ZV9uYW1lcyksCiAgICAgICAgbGFiZWxfbmFtZXM9bnAuYXNhcnJheShwaGFzZXMpLAogICAgICAgIHNlcXVlbmNlX2xlbmd0aD1ucC5hc2FycmF5KGFyZ3Muc2VxdWVuY2VfbGVuZ3RoKSwKICAgICAgICBwaGFzZV9tYXBwaW5nc19qc29uPW5wLmFzYXJyYXkoanNvbi5kdW1wcyhzdXBwb3J0ZWQpKSwKICAgICAgICBzY2hlbWFfdmVyc2lvbj1ucC5hc2FycmF5KCIyLjAiKSwKICAgICkKICAgIHByaW50KAogICAgICAgIGYiV3JvdGUge2xlbihmZWF0dXJlcyl9IHdpbmRvd3MgYWNyb3NzICIKICAgICAgICBmIntsZW4oc2V0KGdyb3VwcykpfSBzZXNzaW9ucyBhbmQge2xlbihvYnNlcnZlZF90ZWNobmlxdWVzKX0gdGVjaG5pcXVlcyIKICAgICkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==","train_phase_model.py":"IiIiVHJhaW4gYW5kIGV4cG9ydCBhIGNvbXBhY3QgU1QtR0NOIHRlbXBvcmFsIHN0YXRlIGNsYXNzaWZpZXIuCgpUaGUgbW9kZWwgcHJlZGljdHMgc3RhdGUgcHJvYmFiaWxpdGllcyBvbmx5LiBSZXBldGl0aW9uIGNvdW50aW5nIGFuZCBsZWdhbApvcmRlcmluZyByZW1haW4gdGhlIHJlc3BvbnNpYmlsaXR5IG9mIHRoZSBkZXRlcm1pbmlzdGljIHRlbXBvcmFsIGRlY29kZXIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCByYW5kb20KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBjbGFzc2lmaWNhdGlvbl9yZXBvcnQsIGYxX3Njb3JlCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IEdyb3VwU2h1ZmZsZVNwbGl0CmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlciwgVGVuc29yRGF0YXNldAoKUE9TRV9FREdFUyA9IFsKICAgICgwLCAxKSwgKDEsIDIpLCAoMiwgMyksICgzLCA3KSwgKDAsIDQpLCAoNCwgNSksICg1LCA2KSwgKDYsIDgpLAogICAgKDksIDEwKSwgKDExLCAxMiksICgxMSwgMTMpLCAoMTMsIDE1KSwgKDE1LCAxNyksICgxNSwgMTkpLAogICAgKDE1LCAyMSksICgxNywgMTkpLCAoMTIsIDE0KSwgKDE0LCAxNiksICgxNiwgMTgpLCAoMTYsIDIwKSwKICAgICgxNiwgMjIpLCAoMTgsIDIwKSwgKDExLCAyMyksICgxMiwgMjQpLCAoMjMsIDI0KSwgKDIzLCAyNSksCiAgICAoMjUsIDI3KSwgKDI3LCAyOSksICgyOSwgMzEpLCAoMjcsIDMxKSwgKDI0LCAyNiksICgyNiwgMjgpLAogICAgKDI4LCAzMCksICgzMCwgMzIpLCAoMjgsIDMyKSwKXQoKCmRlZiBwYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD02MCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sZWFybmluZy1yYXRlIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zZS00KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1oaWRkZW4tc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTk2KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kcm9wb3V0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcGF0aWVuY2UiLCB0eXBlPWludCwgZGVmYXVsdD0xMCkKICAgIHJldHVybiBwYXJzZXIucGFyc2VfYXJncygpCgoKZGVmIHNlZWRfZXZlcnl0aGluZyhzZWVkOiBpbnQpIC0+IE5vbmU6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQoKCmRlZiBub3JtYWxpemVkX2FkamFjZW5jeShqb2ludHM6IGludCA9IDMzKSAtPiB0b3JjaC5UZW5zb3I6CiAgICBhZGphY2VuY3kgPSBucC5leWUoam9pbnRzLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgZm9yIGxlZnQsIHJpZ2h0IGluIFBPU0VfRURHRVM6CiAgICAgICAgYWRqYWNlbmN5W2xlZnQsIHJpZ2h0XSA9IDEKICAgICAgICBhZGphY2VuY3lbcmlnaHQsIGxlZnRdID0gMQogICAgZGVncmVlID0gYWRqYWNlbmN5LnN1bShheGlzPTEpCiAgICBpbnZfc3FydCA9IG5wLmRpYWcobnAucG93ZXIobnAubWF4aW11bShkZWdyZWUsIDEpLCAtMC41KSkKICAgIHJldHVybiB0b3JjaC50ZW5zb3IoaW52X3NxcnQgQCBhZGphY2VuY3kgQCBpbnZfc3FydCwgZHR5cGU9dG9yY2guZmxvYXQzMikKCgpjbGFzcyBTcGF0aWFsR3JhcGhCbG9jayhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlucHV0X3NpemU6IGludCwgb3V0cHV0X3NpemU6IGludCwgZHJvcG91dDogZmxvYXQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucHJvamVjdCA9IG5uLkxpbmVhcihpbnB1dF9zaXplLCBvdXRwdXRfc2l6ZSkKICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0ob3V0cHV0X3NpemUpCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYuYWN0aXZhdGlvbiA9IG5uLkdFTFUoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlucHV0czogdG9yY2guVGVuc29yLCBhZGphY2VuY3k6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGFnZ3JlZ2F0ZWQgPSB0b3JjaC5laW5zdW0oInZ3LGJ0d2MtPmJ0dmMiLCBhZGphY2VuY3ksIGlucHV0cykKICAgICAgICByZXR1cm4gc2VsZi5kcm9wb3V0KHNlbGYuYWN0aXZhdGlvbihzZWxmLm5vcm0oc2VsZi5wcm9qZWN0KGFnZ3JlZ2F0ZWQpKSkpCgoKY2xhc3MgVGVtcG9yYWxQaGFzZVNUR0NOKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2xhc3NlczogaW50LCBoaWRkZW5fc2l6ZTogaW50ID0gOTYsIGRyb3BvdXQ6IGZsb2F0ID0gMC4yKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiYWRqYWNlbmN5Iiwgbm9ybWFsaXplZF9hZGphY2VuY3koKSkKICAgICAgICBzZWxmLnNwYXRpYWwxID0gU3BhdGlhbEdyYXBoQmxvY2soNCwgaGlkZGVuX3NpemUgLy8gMiwgZHJvcG91dCkKICAgICAgICBzZWxmLnNwYXRpYWwyID0gU3BhdGlhbEdyYXBoQmxvY2soaGlkZGVuX3NpemUgLy8gMiwgaGlkZGVuX3NpemUsIGRyb3BvdXQpCiAgICAgICAgc2VsZi50ZW1wb3JhbCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYxZChoaWRkZW5fc2l6ZSwgaGlkZGVuX3NpemUsIGtlcm5lbF9zaXplPTUsIHBhZGRpbmc9MiksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTFkKGhpZGRlbl9zaXplKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICBubi5Db252MWQoCiAgICAgICAgICAgICAgICBoaWRkZW5fc2l6ZSwKICAgICAgICAgICAgICAgIGhpZGRlbl9zaXplLAogICAgICAgICAgICAgICAga2VybmVsX3NpemU9NSwKICAgICAgICAgICAgICAgIHBhZGRpbmc9NCwKICAgICAgICAgICAgICAgIGRpbGF0aW9uPTIsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTFkKGhpZGRlbl9zaXplKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICkKICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5MaW5lYXIoaGlkZGVuX3NpemUsIGNsYXNzZXMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW5wdXRzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAjIGlucHV0czogW2JhdGNoLCB0aW1lLCBqb2ludCwgY2hhbm5lbF0KICAgICAgICBzcGF0aWFsID0gc2VsZi5zcGF0aWFsMShpbnB1dHMsIHNlbGYuYWRqYWNlbmN5KQogICAgICAgIHNwYXRpYWwgPSBzZWxmLnNwYXRpYWwyKHNwYXRpYWwsIHNlbGYuYWRqYWNlbmN5KQogICAgICAgIHZpc2liaWxpdHkgPSBpbnB1dHNbLi4uLCAzXS5jbGFtcCgwLCAxKS51bnNxdWVlemUoLTEpCiAgICAgICAgcG9vbGVkID0gKHNwYXRpYWwgKiB2aXNpYmlsaXR5KS5zdW0oZGltPTIpIC8gdmlzaWJpbGl0eS5zdW0oZGltPTIpLmNsYW1wX21pbigxKQogICAgICAgIHRlbXBvcmFsID0gc2VsZi50ZW1wb3JhbChwb29sZWQudHJhbnNwb3NlKDEsIDIpKS50cmFuc3Bvc2UoMSwgMikKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHRlbXBvcmFsKQoKCmRlZiBzcGxpdF9ieV9zZXNzaW9uKGdyb3VwczogbnAubmRhcnJheSwgc2VlZDogaW50KSAtPiB0dXBsZVtucC5uZGFycmF5LCAuLi5dOgogICAgaW5kZXhlcyA9IG5wLmFyYW5nZShsZW4oZ3JvdXBzKSkKICAgIGZpcnN0ID0gR3JvdXBTaHVmZmxlU3BsaXQobl9zcGxpdHM9MSwgdGVzdF9zaXplPTAuMzAsIHJhbmRvbV9zdGF0ZT1zZWVkKQogICAgdHJhaW5faW5kZXhlcywgaG9sZG91dF9pbmRleGVzID0gbmV4dChmaXJzdC5zcGxpdChpbmRleGVzLCBncm91cHM9Z3JvdXBzKSkKICAgIGhvbGRvdXRfZ3JvdXBzID0gZ3JvdXBzW2hvbGRvdXRfaW5kZXhlc10KICAgIHNlY29uZCA9IEdyb3VwU2h1ZmZsZVNwbGl0KG5fc3BsaXRzPTEsIHRlc3Rfc2l6ZT0wLjUwLCByYW5kb21fc3RhdGU9c2VlZCArIDEpCiAgICB2YWxpZGF0aW9uX2xvY2FsLCB0ZXN0X2xvY2FsID0gbmV4dCgKICAgICAgICBzZWNvbmQuc3BsaXQoaG9sZG91dF9pbmRleGVzLCBncm91cHM9aG9sZG91dF9ncm91cHMpCiAgICApCiAgICByZXR1cm4gKAogICAgICAgIHRyYWluX2luZGV4ZXMsCiAgICAgICAgaG9sZG91dF9pbmRleGVzW3ZhbGlkYXRpb25fbG9jYWxdLAogICAgICAgIGhvbGRvdXRfaW5kZXhlc1t0ZXN0X2xvY2FsXSwKICAgICkKCgpkZWYgbG9hZGVyX2ZvcigKICAgIGZlYXR1cmVzOiBucC5uZGFycmF5LAogICAgbGFiZWxzOiBucC5uZGFycmF5LAogICAgbWFza3M6IG5wLm5kYXJyYXksCiAgICBpbmRleGVzOiBucC5uZGFycmF5LAogICAgYmF0Y2hfc2l6ZTogaW50LAogICAgc2h1ZmZsZTogYm9vbCwKKSAtPiBEYXRhTG9hZGVyOgogICAgZGF0YXNldCA9IFRlbnNvckRhdGFzZXQoCiAgICAgICAgdG9yY2guZnJvbV9udW1weShmZWF0dXJlc1tpbmRleGVzXSkuZmxvYXQoKSwKICAgICAgICB0b3JjaC5mcm9tX251bXB5KGxhYmVsc1tpbmRleGVzXSkubG9uZygpLAogICAgICAgIHRvcmNoLmZyb21fbnVtcHkobWFza3NbaW5kZXhlc10pLmJvb2woKSwKICAgICkKICAgIHJldHVybiBEYXRhTG9hZGVyKGRhdGFzZXQsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgc2h1ZmZsZT1zaHVmZmxlKQoKCmRlZiBjbGFzc193ZWlnaHRzKAogICAgbGFiZWxzOiBucC5uZGFycmF5LCBtYXNrczogbnAubmRhcnJheSwgaW5kZXhlczogbnAubmRhcnJheSwgY2xhc3NlczogaW50CikgLT4gdG9yY2guVGVuc29yOgogICAgdmFsdWVzID0gbGFiZWxzW2luZGV4ZXNdW21hc2tzW2luZGV4ZXNdXQogICAgY291bnRzID0gbnAuYmluY291bnQodmFsdWVzLCBtaW5sZW5ndGg9Y2xhc3NlcykuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICB3ZWlnaHRzID0gY291bnRzLnN1bSgpIC8gbnAubWF4aW11bShjb3VudHMsIDEpCiAgICB3ZWlnaHRzID0gd2VpZ2h0cyAvIG1heCh3ZWlnaHRzLm1lYW4oKSwgMWUtNikKICAgIHdlaWdodHNbMF0gPSAwCiAgICByZXR1cm4gdG9yY2gudGVuc29yKHdlaWdodHMsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgZXZhbHVhdGUoCiAgICBtb2RlbDogbm4uTW9kdWxlLAogICAgbG9hZGVyOiBEYXRhTG9hZGVyLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UsCiAgICBwYWRfaWQ6IGludCwKKSAtPiB0dXBsZVtmbG9hdCwgbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICBtb2RlbC5ldmFsKCkKICAgIHByZWRpY3Rpb25zOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgIHRhcmdldHM6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgZm9yIGZlYXR1cmVzLCBsYWJlbHMsIG1hc2tzIGluIGxvYWRlcjoKICAgICAgICBsb2dpdHMgPSBtb2RlbChmZWF0dXJlcy50byhkZXZpY2UpKQogICAgICAgIHZhbGlkID0gbWFza3MudG8oZGV2aWNlKSAmIGxhYmVscy50byhkZXZpY2UpLm5lKHBhZF9pZCkKICAgICAgICBwcmVkaWN0aW9ucy5hcHBlbmQobG9naXRzLmFyZ21heChkaW09LTEpW3ZhbGlkXS5jcHUoKS5udW1weSgpKQogICAgICAgIHRhcmdldHMuYXBwZW5kKGxhYmVscy50byhkZXZpY2UpW3ZhbGlkXS5jcHUoKS5udW1weSgpKQogICAgeV9wcmVkID0gbnAuY29uY2F0ZW5hdGUocHJlZGljdGlvbnMpIGlmIHByZWRpY3Rpb25zIGVsc2UgbnAuYXJyYXkoW10pCiAgICB5X3RydWUgPSBucC5jb25jYXRlbmF0ZSh0YXJnZXRzKSBpZiB0YXJnZXRzIGVsc2UgbnAuYXJyYXkoW10pCiAgICBzY29yZSA9ICgKICAgICAgICBmMV9zY29yZSh5X3RydWUsIHlfcHJlZCwgYXZlcmFnZT0ibWFjcm8iLCB6ZXJvX2RpdmlzaW9uPTApCiAgICAgICAgaWYgbGVuKHlfdHJ1ZSkKICAgICAgICBlbHNlIDAuMAogICAgKQogICAgcmV0dXJuIGZsb2F0KHNjb3JlKSwgeV90cnVlLCB5X3ByZWQKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcmdzID0gcGFyc2VfYXJncygpCiAgICBzZWVkX2V2ZXJ5dGhpbmcoYXJncy5zZWVkKQogICAgZGF0YSA9IG5wLmxvYWQoYXJncy5kYXRhc2V0LCBhbGxvd19waWNrbGU9RmFsc2UpCiAgICBmZWF0dXJlcyA9IGRhdGFbImZlYXR1cmVzIl0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBsYWJlbHMgPSBkYXRhWyJsYWJlbHMiXS5hc3R5cGUobnAuaW50NjQpCiAgICBtYXNrcyA9IGRhdGFbIm1hc2siXS5hc3R5cGUoYm9vbCkKICAgIGdyb3VwcyA9IGRhdGFbImdyb3VwcyJdLmFzdHlwZShzdHIpCiAgICBvcmlnaW5zID0gKAogICAgICAgIGRhdGFbIm9yaWdpbnMiXS5hc3R5cGUoc3RyKQogICAgICAgIGlmICJvcmlnaW5zIiBpbiBkYXRhLmZpbGVzCiAgICAgICAgZWxzZSBucC5mdWxsKGxlbihncm91cHMpLCAicmVhbCIpCiAgICApCiAgICBsYWJlbF9uYW1lcyA9IGRhdGFbImxhYmVsX25hbWVzIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgIGlmIGxlbihzZXQoZ3JvdXBzKSkgPCA0OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgIkF0IGxlYXN0IGZvdXIgaW5kZXBlbmRlbnQgc2Vzc2lvbnMgYXJlIHJlcXVpcmVkLiBVc2UgbW9yZSBzZXNzaW9ucyAiCiAgICAgICAgICAgICJiZWZvcmUgdHJ1c3RpbmcgdmFsaWRhdGlvbiByZXN1bHRzLiIKICAgICAgICApCgogICAgc3ludGhldGljX2lkcyA9IG5wLmZsYXRub256ZXJvKG9yaWdpbnMgPT0gInN5bnRoZXRpYyIpCiAgICByZWFsX2lkcyA9IG5wLmZsYXRub256ZXJvKG9yaWdpbnMgIT0gInN5bnRoZXRpYyIpCiAgICBpZiBsZW4oc3ludGhldGljX2lkcykgYW5kIGxlbihzZXQoZ3JvdXBzW3JlYWxfaWRzXSkpID49IDQ6CiAgICAgICAgcmVhbF90cmFpbiwgcmVhbF92YWxpZGF0aW9uLCByZWFsX3Rlc3QgPSBzcGxpdF9ieV9zZXNzaW9uKAogICAgICAgICAgICBncm91cHNbcmVhbF9pZHNdLCBhcmdzLnNlZWQKICAgICAgICApCiAgICAgICAgdHJhaW5faWRzID0gbnAuY29uY2F0ZW5hdGUoW3JlYWxfaWRzW3JlYWxfdHJhaW5dLCBzeW50aGV0aWNfaWRzXSkKICAgICAgICB2YWxpZGF0aW9uX2lkcyA9IHJlYWxfaWRzW3JlYWxfdmFsaWRhdGlvbl0KICAgICAgICB0ZXN0X2lkcyA9IHJlYWxfaWRzW3JlYWxfdGVzdF0KICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJzcGxpdDogdHJhaW4gaW5jbHVkZXMge2xlbihzeW50aGV0aWNfaWRzKX0gc3ludGhldGljIHdpbmRvd3M7ICIKICAgICAgICAgICAgInZhbGlkYXRpb24gYW5kIHRlc3QgYXJlIHJlYWwtb25seSIKICAgICAgICApCiAgICBlbHNlOgogICAgICAgIHRyYWluX2lkcywgdmFsaWRhdGlvbl9pZHMsIHRlc3RfaWRzID0gc3BsaXRfYnlfc2Vzc2lvbihncm91cHMsIGFyZ3Muc2VlZCkKICAgICAgICBpZiBsZW4oc3ludGhldGljX2lkcyk6CiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgIndhcm5pbmc6IG5vIHN1ZmZpY2llbnQgcmVhbCBzZXNzaW9uczsgdmFsaWRhdGlvbiBpcyBzeW50aGV0aWMgIgogICAgICAgICAgICAgICAgImFuZCBpcyBvbmx5IGEgcGlwZWxpbmUgY2hlY2siCiAgICAgICAgICAgICkKICAgIHRyYWluX2xvYWRlciA9IGxvYWRlcl9mb3IoCiAgICAgICAgZmVhdHVyZXMsIGxhYmVscywgbWFza3MsIHRyYWluX2lkcywgYXJncy5iYXRjaF9zaXplLCBUcnVlCiAgICApCiAgICB2YWxpZGF0aW9uX2xvYWRlciA9IGxvYWRlcl9mb3IoCiAgICAgICAgZmVhdHVyZXMsIGxhYmVscywgbWFza3MsIHZhbGlkYXRpb25faWRzLCBhcmdzLmJhdGNoX3NpemUsIEZhbHNlCiAgICApCiAgICB0ZXN0X2xvYWRlciA9IGxvYWRlcl9mb3IoCiAgICAgICAgZmVhdHVyZXMsIGxhYmVscywgbWFza3MsIHRlc3RfaWRzLCBhcmdzLmJhdGNoX3NpemUsIEZhbHNlCiAgICApCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsID0gVGVtcG9yYWxQaGFzZVNUR0NOKAogICAgICAgIGxlbihsYWJlbF9uYW1lcyksIGFyZ3MuaGlkZGVuX3NpemUsIGFyZ3MuZHJvcG91dAogICAgKS50byhkZXZpY2UpCiAgICB3ZWlnaHRzID0gY2xhc3Nfd2VpZ2h0cygKICAgICAgICBsYWJlbHMsIG1hc2tzLCB0cmFpbl9pZHMsIGxlbihsYWJlbF9uYW1lcykKICAgICkudG8oZGV2aWNlKQogICAgY3JpdGVyaW9uID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcyh3ZWlnaHQ9d2VpZ2h0cywgaWdub3JlX2luZGV4PTApCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVygKICAgICAgICBtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWFyZ3MubGVhcm5pbmdfcmF0ZSwgd2VpZ2h0X2RlY2F5PTFlLTQKICAgICkKCiAgICBhcmdzLm91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgY2hlY2twb2ludF9wYXRoID0gYXJncy5vdXRwdXRfZGlyIC8gImJlc3RfdGVtcG9yYWxfcGhhc2UucHQiCiAgICBiZXN0X3Njb3JlID0gLTEuMAogICAgZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgPSAwCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoMSwgYXJncy5lcG9jaHMgKyAxKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgcnVubmluZ19sb3NzID0gMC4wCiAgICAgICAgYmF0Y2hlcyA9IDAKICAgICAgICBmb3IgYmF0Y2hfZmVhdHVyZXMsIGJhdGNoX2xhYmVscywgYmF0Y2hfbWFza3MgaW4gdHJhaW5fbG9hZGVyOgogICAgICAgICAgICBiYXRjaF9mZWF0dXJlcyA9IGJhdGNoX2ZlYXR1cmVzLnRvKGRldmljZSkKICAgICAgICAgICAgYmF0Y2hfbGFiZWxzID0gYmF0Y2hfbGFiZWxzLnRvKGRldmljZSkKICAgICAgICAgICAgYmF0Y2hfbWFza3MgPSBiYXRjaF9tYXNrcy50byhkZXZpY2UpCiAgICAgICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoYmF0Y2hfZmVhdHVyZXMpCiAgICAgICAgICAgIHRhcmdldHMgPSBiYXRjaF9sYWJlbHMubWFza2VkX2ZpbGwofmJhdGNoX21hc2tzLCAwKQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKAogICAgICAgICAgICAgICAgbG9naXRzLnJlc2hhcGUoLTEsIGxlbihsYWJlbF9uYW1lcykpLCB0YXJnZXRzLnJlc2hhcGUoLTEpCiAgICAgICAgICAgICkKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIDEuMCkKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgICAgICBydW5uaW5nX2xvc3MgKz0gZmxvYXQobG9zcy5pdGVtKCkpCiAgICAgICAgICAgIGJhdGNoZXMgKz0gMQogICAgICAgIHZhbGlkYXRpb25fZjEsIF8sIF8gPSBldmFsdWF0ZShtb2RlbCwgdmFsaWRhdGlvbl9sb2FkZXIsIGRldmljZSwgMCkKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJlcG9jaD17ZXBvY2g6MDNkfSBsb3NzPXtydW5uaW5nX2xvc3MgLyBtYXgoYmF0Y2hlcywgMSk6LjRmfSAiCiAgICAgICAgICAgIGYidmFsaWRhdGlvbl9tYWNyb19mMT17dmFsaWRhdGlvbl9mMTouNGZ9IgogICAgICAgICkKICAgICAgICBpZiB2YWxpZGF0aW9uX2YxID4gYmVzdF9zY29yZToKICAgICAgICAgICAgYmVzdF9zY29yZSA9IHZhbGlkYXRpb25fZjEKICAgICAgICAgICAgZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgPSAwCiAgICAgICAgICAgIHRvcmNoLnNhdmUobW9kZWwuc3RhdGVfZGljdCgpLCBjaGVja3BvaW50X3BhdGgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgKz0gMQogICAgICAgICAgICBpZiBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA+PSBhcmdzLnBhdGllbmNlOgogICAgICAgICAgICAgICAgcHJpbnQoIkVhcmx5IHN0b3BwaW5nIikKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQoY2hlY2twb2ludF9wYXRoLCBtYXBfbG9jYXRpb249ZGV2aWNlKSkKICAgIHRlc3RfZjEsIHlfdHJ1ZSwgeV9wcmVkID0gZXZhbHVhdGUobW9kZWwsIHRlc3RfbG9hZGVyLCBkZXZpY2UsIDApCiAgICByZXBvcnQgPSBjbGFzc2lmaWNhdGlvbl9yZXBvcnQoCiAgICAgICAgeV90cnVlLAogICAgICAgIHlfcHJlZCwKICAgICAgICBsYWJlbHM9bGlzdChyYW5nZSgxLCBsZW4obGFiZWxfbmFtZXMpKSksCiAgICAgICAgdGFyZ2V0X25hbWVzPWxhYmVsX25hbWVzWzE6XSwKICAgICAgICB6ZXJvX2RpdmlzaW9uPTAsCiAgICAgICAgb3V0cHV0X2RpY3Q9VHJ1ZSwKICAgICkKICAgIHByaW50KGYidGVzdF9tYWNyb19mMT17dGVzdF9mMTouNGZ9IikKCiAgICBtb2RlbCA9IG1vZGVsLmNwdSgpLmV2YWwoKQogICAgc2VxdWVuY2VfbGVuZ3RoID0gaW50KGRhdGFbInNlcXVlbmNlX2xlbmd0aCJdKQogICAgZHVtbXkgPSB0b3JjaC56ZXJvcygxLCBzZXF1ZW5jZV9sZW5ndGgsIDMzLCA0LCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgb25ueF9wYXRoID0gYXJncy5vdXRwdXRfZGlyIC8gInRlbXBvcmFsX3BoYXNlX2NsYXNzaWZpZXIub25ueCIKICAgIHRvcmNoLm9ubnguZXhwb3J0KAogICAgICAgIG1vZGVsLAogICAgICAgIGR1bW15LAogICAgICAgIG9ubnhfcGF0aCwKICAgICAgICBpbnB1dF9uYW1lcz1bImxhbmRtYXJrcyJdLAogICAgICAgIG91dHB1dF9uYW1lcz1bInN0YXRlX2xvZ2l0cyJdLAogICAgICAgIGR5bmFtaWNfYXhlcz17CiAgICAgICAgICAgICJsYW5kbWFya3MiOiB7MDogImJhdGNoIiwgMTogInRpbWUifSwKICAgICAgICAgICAgInN0YXRlX2xvZ2l0cyI6IHswOiAiYmF0Y2giLCAxOiAidGltZSJ9LAogICAgICAgIH0sCiAgICAgICAgb3BzZXRfdmVyc2lvbj0xOCwKICAgICkKICAgIG1ldGFkYXRhID0gewogICAgICAgICJzY2hlbWFfdmVyc2lvbiI6ICIxLjAiLAogICAgICAgICJtb2RlbF90eXBlIjogInRlbXBvcmFsLXN0YXRlLWVtaXNzaW9uIiwKICAgICAgICAiYXJjaGl0ZWN0dXJlIjogImNvbXBhY3Qtc3RnY24tdGNuIiwKICAgICAgICAiaW5wdXQiOiB7CiAgICAgICAgICAgICJuYW1lIjogImxhbmRtYXJrcyIsCiAgICAgICAgICAgICJsYXlvdXQiOiAiQlRWQyIsCiAgICAgICAgICAgICJqb2ludHMiOiAzMywKICAgICAgICAgICAgImNoYW5uZWxzIjogWyJ4IiwgInkiLCAieiIsICJ2aXNpYmlsaXR5Il0sCiAgICAgICAgICAgICJzZXF1ZW5jZV9sZW5ndGgiOiBzZXF1ZW5jZV9sZW5ndGgsCiAgICAgICAgICAgICJub3JtYWxpemF0aW9uIjogImhpcC1jZW50ZXJlZC10b3Jzby1zY2FsZSIKICAgICAgICB9LAogICAgICAgICJvdXRwdXQiOiB7CiAgICAgICAgICAgICJuYW1lIjogInN0YXRlX2xvZ2l0cyIsCiAgICAgICAgICAgICJsYXlvdXQiOiAiQlRDIiwKICAgICAgICAgICAgImxhYmVscyI6IGxhYmVsX25hbWVzCiAgICAgICAgfSwKICAgICAgICAidmFsaWRhdGlvbiI6IHsKICAgICAgICAgICAgInNwbGl0X3VuaXQiOiAic2Vzc2lvbiIsCiAgICAgICAgICAgICJ2YWxpZGF0aW9uX21hY3JvX2YxIjogYmVzdF9zY29yZSwKICAgICAgICAgICAgInRlc3RfbWFjcm9fZjEiOiB0ZXN0X2YxLAogICAgICAgICAgICAidHJhaW5fc2Vzc2lvbnMiOiBpbnQobGVuKHNldChncm91cHNbdHJhaW5faWRzXSkpKSwKICAgICAgICAgICAgInZhbGlkYXRpb25fc2Vzc2lvbnMiOiBpbnQobGVuKHNldChncm91cHNbdmFsaWRhdGlvbl9pZHNdKSkpLAogICAgICAgICAgICAidGVzdF9zZXNzaW9ucyI6IGludChsZW4oc2V0KGdyb3Vwc1t0ZXN0X2lkc10pKSkKICAgICAgICB9CiAgICB9CiAgICAoYXJncy5vdXRwdXRfZGlyIC8gInRlbXBvcmFsX3BoYXNlX2NsYXNzaWZpZXIubWV0YWRhdGEuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhtZXRhZGF0YSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICAoYXJncy5vdXRwdXRfZGlyIC8gInRlc3RfcmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHByaW50KGYiRXhwb3J0ZWQge29ubnhfcGF0aH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK","train_universal_model.py":"IiIiVHJhaW4gYW5kIGV4cG9ydCBvbmUgdGVjaG5pcXVlLWNvbmRpdGlvbmVkIHRlbXBvcmFsIHBoYXNlIG1vZGVsLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCByYW5kb20KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBjbGFzc2lmaWNhdGlvbl9yZXBvcnQsIGYxX3Njb3JlCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IEdyb3VwU2h1ZmZsZVNwbGl0CmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlciwgVGVuc29yRGF0YXNldAoKZnJvbSB0cmFpbl9waGFzZV9tb2RlbCBpbXBvcnQgU3BhdGlhbEdyYXBoQmxvY2ssIG5vcm1hbGl6ZWRfYWRqYWNlbmN5CgoKY2xhc3MgVW5pdmVyc2FsVGVtcG9yYWxTVEdDTihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgY2xhc3NlczogaW50LAogICAgICAgIHRlY2huaXF1ZXM6IGludCwKICAgICAgICBoaWRkZW5fc2l6ZTogaW50ID0gOTYsCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSAwLjIsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJhZGphY2VuY3kiLCBub3JtYWxpemVkX2FkamFjZW5jeSgpKQogICAgICAgIHNlbGYuc3BhdGlhbDEgPSBTcGF0aWFsR3JhcGhCbG9jayg0LCBoaWRkZW5fc2l6ZSAvLyAyLCBkcm9wb3V0KQogICAgICAgIHNlbGYuc3BhdGlhbDIgPSBTcGF0aWFsR3JhcGhCbG9jayhoaWRkZW5fc2l6ZSAvLyAyLCBoaWRkZW5fc2l6ZSwgZHJvcG91dCkKICAgICAgICBzZWxmLnRlY2huaXF1ZV9lbWJlZGRpbmcgPSBubi5FbWJlZGRpbmcodGVjaG5pcXVlcywgaGlkZGVuX3NpemUpCiAgICAgICAgc2VsZi50ZW1wb3JhbCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYxZChoaWRkZW5fc2l6ZSwgaGlkZGVuX3NpemUsIDUsIHBhZGRpbmc9MiksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTFkKGhpZGRlbl9zaXplKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICBubi5Db252MWQoaGlkZGVuX3NpemUsIGhpZGRlbl9zaXplLCA1LCBwYWRkaW5nPTQsIGRpbGF0aW9uPTIpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0xZChoaWRkZW5fc2l6ZSksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uRHJvcG91dChkcm9wb3V0KSwKICAgICAgICApCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uTGluZWFyKGhpZGRlbl9zaXplLCBjbGFzc2VzKQoKICAgIGRlZiBmb3J3YXJkKAogICAgICAgIHNlbGYsIGxhbmRtYXJrczogdG9yY2guVGVuc29yLCB0ZWNobmlxdWVfaWQ6IHRvcmNoLlRlbnNvcgogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgc3BhdGlhbCA9IHNlbGYuc3BhdGlhbDEobGFuZG1hcmtzLCBzZWxmLmFkamFjZW5jeSkKICAgICAgICBzcGF0aWFsID0gc2VsZi5zcGF0aWFsMihzcGF0aWFsLCBzZWxmLmFkamFjZW5jeSkKICAgICAgICB2aXNpYmlsaXR5ID0gbGFuZG1hcmtzWy4uLiwgM10uY2xhbXAoMCwgMSkudW5zcXVlZXplKC0xKQogICAgICAgIHBvb2xlZCA9IChzcGF0aWFsICogdmlzaWJpbGl0eSkuc3VtKGRpbT0yKSAvIHZpc2liaWxpdHkuc3VtKAogICAgICAgICAgICBkaW09MgogICAgICAgICkuY2xhbXBfbWluKDEpCiAgICAgICAgY29uZGl0aW9uZWQgPSBwb29sZWQgKyBzZWxmLnRlY2huaXF1ZV9lbWJlZGRpbmcodGVjaG5pcXVlX2lkKS51bnNxdWVlemUoMSkKICAgICAgICB0ZW1wb3JhbCA9IHNlbGYudGVtcG9yYWwoY29uZGl0aW9uZWQudHJhbnNwb3NlKDEsIDIpKS50cmFuc3Bvc2UoMSwgMikKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHRlbXBvcmFsKQoKCmRlZiBwYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD02MCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sZWFybmluZy1yYXRlIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zZS00KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1oaWRkZW4tc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTk2KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kcm9wb3V0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcGF0aWVuY2UiLCB0eXBlPWludCwgZGVmYXVsdD0xMCkKICAgIHJldHVybiBwYXJzZXIucGFyc2VfYXJncygpCgoKZGVmIHNwbGl0X2dyb3Vwcyhncm91cHM6IG5wLm5kYXJyYXksIHNlZWQ6IGludCkgLT4gdHVwbGVbbnAubmRhcnJheSwgLi4uXToKICAgIGluZGV4ZXMgPSBucC5hcmFuZ2UobGVuKGdyb3VwcykpCiAgICBmaXJzdCA9IEdyb3VwU2h1ZmZsZVNwbGl0KG5fc3BsaXRzPTEsIHRlc3Rfc2l6ZT0wLjMwLCByYW5kb21fc3RhdGU9c2VlZCkKICAgIHRyYWluLCBob2xkb3V0ID0gbmV4dChmaXJzdC5zcGxpdChpbmRleGVzLCBncm91cHM9Z3JvdXBzKSkKICAgIHNlY29uZCA9IEdyb3VwU2h1ZmZsZVNwbGl0KG5fc3BsaXRzPTEsIHRlc3Rfc2l6ZT0wLjUwLCByYW5kb21fc3RhdGU9c2VlZCArIDEpCiAgICB2YWxpZGF0aW9uX2xvY2FsLCB0ZXN0X2xvY2FsID0gbmV4dCgKICAgICAgICBzZWNvbmQuc3BsaXQoaG9sZG91dCwgZ3JvdXBzPWdyb3Vwc1tob2xkb3V0XSkKICAgICkKICAgIHJldHVybiB0cmFpbiwgaG9sZG91dFt2YWxpZGF0aW9uX2xvY2FsXSwgaG9sZG91dFt0ZXN0X2xvY2FsXQoKCmRlZiBtYWtlX2xvYWRlcihkYXRhOiBkaWN0LCBpbmRleGVzLCBiYXRjaF9zaXplOiBpbnQsIHNodWZmbGU6IGJvb2wpIC0+IERhdGFMb2FkZXI6CiAgICBkYXRhc2V0ID0gVGVuc29yRGF0YXNldCgKICAgICAgICB0b3JjaC5mcm9tX251bXB5KGRhdGFbImZlYXR1cmVzIl1baW5kZXhlc10pLmZsb2F0KCksCiAgICAgICAgdG9yY2guZnJvbV9udW1weShkYXRhWyJ0ZWNobmlxdWVfaWRzIl1baW5kZXhlc10pLmxvbmcoKSwKICAgICAgICB0b3JjaC5mcm9tX251bXB5KGRhdGFbImxhYmVscyJdW2luZGV4ZXNdKS5sb25nKCksCiAgICAgICAgdG9yY2guZnJvbV9udW1weShkYXRhWyJtYXNrIl1baW5kZXhlc10pLmJvb2woKSwKICAgICkKICAgIHJldHVybiBEYXRhTG9hZGVyKGRhdGFzZXQsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgc2h1ZmZsZT1zaHVmZmxlKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGV2YWx1YXRlKG1vZGVsLCBsb2FkZXIsIGRldmljZSkgLT4gdHVwbGVbZmxvYXQsIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgbW9kZWwuZXZhbCgpCiAgICBwcmVkaWN0aW9ucywgdGFyZ2V0cyA9IFtdLCBbXQogICAgZm9yIGxhbmRtYXJrcywgdGVjaG5pcXVlX2lkcywgbGFiZWxzLCBtYXNrIGluIGxvYWRlcjoKICAgICAgICBsb2dpdHMgPSBtb2RlbChsYW5kbWFya3MudG8oZGV2aWNlKSwgdGVjaG5pcXVlX2lkcy50byhkZXZpY2UpKQogICAgICAgIHZhbGlkID0gbWFzay50byhkZXZpY2UpICYgbGFiZWxzLnRvKGRldmljZSkubmUoMCkKICAgICAgICBwcmVkaWN0aW9ucy5hcHBlbmQobG9naXRzLmFyZ21heCgtMSlbdmFsaWRdLmNwdSgpLm51bXB5KCkpCiAgICAgICAgdGFyZ2V0cy5hcHBlbmQobGFiZWxzLnRvKGRldmljZSlbdmFsaWRdLmNwdSgpLm51bXB5KCkpCiAgICBwcmVkaWN0ZWQgPSBucC5jb25jYXRlbmF0ZShwcmVkaWN0aW9ucykgaWYgcHJlZGljdGlvbnMgZWxzZSBucC5hcnJheShbXSkKICAgIGFjdHVhbCA9IG5wLmNvbmNhdGVuYXRlKHRhcmdldHMpIGlmIHRhcmdldHMgZWxzZSBucC5hcnJheShbXSkKICAgIHNjb3JlID0gKAogICAgICAgIGYxX3Njb3JlKGFjdHVhbCwgcHJlZGljdGVkLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkKICAgICAgICBpZiBsZW4oYWN0dWFsKQogICAgICAgIGVsc2UgMC4wCiAgICApCiAgICByZXR1cm4gZmxvYXQoc2NvcmUpLCBhY3R1YWwsIHByZWRpY3RlZAoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIGFyZ3MgPSBwYXJzZV9hcmdzKCkKICAgIHJhbmRvbS5zZWVkKGFyZ3Muc2VlZCkKICAgIG5wLnJhbmRvbS5zZWVkKGFyZ3Muc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKGFyZ3Muc2VlZCkKICAgIHJhdyA9IG5wLmxvYWQoYXJncy5kYXRhc2V0LCBhbGxvd19waWNrbGU9RmFsc2UpCiAgICByZXF1aXJlZCA9IHsiZmVhdHVyZXMiLCAidGVjaG5pcXVlX2lkcyIsICJ0ZWNobmlxdWVfbmFtZXMiLCAibGFiZWxzIiwgIm1hc2sifQogICAgbWlzc2luZyA9IHJlcXVpcmVkLmRpZmZlcmVuY2UocmF3LmZpbGVzKQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5pdmVyc2FsIGRhdGFzZXQgbWlzc2luZzoge3NvcnRlZChtaXNzaW5nKX0iKQogICAgZGF0YSA9IHsKICAgICAgICAiZmVhdHVyZXMiOiByYXdbImZlYXR1cmVzIl0uYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICJ0ZWNobmlxdWVfaWRzIjogcmF3WyJ0ZWNobmlxdWVfaWRzIl0uYXN0eXBlKG5wLmludDY0KSwKICAgICAgICAibGFiZWxzIjogcmF3WyJsYWJlbHMiXS5hc3R5cGUobnAuaW50NjQpLAogICAgICAgICJtYXNrIjogcmF3WyJtYXNrIl0uYXN0eXBlKGJvb2wpLAogICAgfQogICAgZ3JvdXBzID0gcmF3WyJncm91cHMiXS5hc3R5cGUoc3RyKQogICAgbGFiZWxfbmFtZXMgPSByYXdbImxhYmVsX25hbWVzIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgIHRlY2huaXF1ZV9uYW1lcyA9IHJhd1sidGVjaG5pcXVlX25hbWVzIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgIGlmIGxlbihzZXQoZ3JvdXBzKSkgPCA0OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQXQgbGVhc3QgZm91ciBpbmRlcGVuZGVudCBzZXNzaW9ucyBhcmUgcmVxdWlyZWQiKQogICAgdHJhaW5faWRzLCB2YWxpZGF0aW9uX2lkcywgdGVzdF9pZHMgPSBzcGxpdF9ncm91cHMoZ3JvdXBzLCBhcmdzLnNlZWQpCiAgICBsb2FkZXJzID0gWwogICAgICAgIG1ha2VfbG9hZGVyKGRhdGEsIGlkcywgYXJncy5iYXRjaF9zaXplLCBzaHVmZmxlKQogICAgICAgIGZvciBpZHMsIHNodWZmbGUgaW4gWwogICAgICAgICAgICAodHJhaW5faWRzLCBUcnVlKSwKICAgICAgICAgICAgKHZhbGlkYXRpb25faWRzLCBGYWxzZSksCiAgICAgICAgICAgICh0ZXN0X2lkcywgRmFsc2UpLAogICAgICAgIF0KICAgIF0KICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbW9kZWwgPSBVbml2ZXJzYWxUZW1wb3JhbFNUR0NOKAogICAgICAgIGxlbihsYWJlbF9uYW1lcyksIGxlbih0ZWNobmlxdWVfbmFtZXMpLCBhcmdzLmhpZGRlbl9zaXplLCBhcmdzLmRyb3BvdXQKICAgICkudG8oZGV2aWNlKQogICAgdHJhaW5fdmFsdWVzID0gZGF0YVsibGFiZWxzIl1bdHJhaW5faWRzXVtkYXRhWyJtYXNrIl1bdHJhaW5faWRzXV0KICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KHRyYWluX3ZhbHVlcywgbWlubGVuZ3RoPWxlbihsYWJlbF9uYW1lcykpLmFzdHlwZShmbG9hdCkKICAgIHdlaWdodHMgPSBjb3VudHMuc3VtKCkgLyBucC5tYXhpbXVtKGNvdW50cywgMSkKICAgIHdlaWdodHMgLz0gbWF4KHdlaWdodHMubWVhbigpLCAxZS02KQogICAgd2VpZ2h0c1swXSA9IDAKICAgIGNyaXRlcmlvbiA9IG5uLkNyb3NzRW50cm9weUxvc3MoCiAgICAgICAgd2VpZ2h0PXRvcmNoLnRlbnNvcih3ZWlnaHRzLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9ZGV2aWNlKSwKICAgICAgICBpZ25vcmVfaW5kZXg9MCwKICAgICkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgIG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9YXJncy5sZWFybmluZ19yYXRlLCB3ZWlnaHRfZGVjYXk9MWUtNAogICAgKQogICAgYXJncy5vdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGNoZWNrcG9pbnQgPSBhcmdzLm91dHB1dF9kaXIgLyAiYmVzdF91bml2ZXJzYWxfdGVtcG9yYWwucHQiCiAgICBiZXN0LCBzdGFsZSA9IC0xLjAsIDAKICAgIGZvciBlcG9jaCBpbiByYW5nZSgxLCBhcmdzLmVwb2NocyArIDEpOgogICAgICAgIG1vZGVsLnRyYWluKCkKICAgICAgICBsb3NzZXMgPSBbXQogICAgICAgIGZvciBsYW5kbWFya3MsIHRlY2huaXF1ZV9pZHMsIGxhYmVscywgbWFzayBpbiBsb2FkZXJzWzBdOgogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGxhbmRtYXJrcy50byhkZXZpY2UpLCB0ZWNobmlxdWVfaWRzLnRvKGRldmljZSkpCiAgICAgICAgICAgIHRhcmdldHMgPSBsYWJlbHMudG8oZGV2aWNlKS5tYXNrZWRfZmlsbCh+bWFzay50byhkZXZpY2UpLCAwKQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKGxvZ2l0cy5mbGF0dGVuKDAsIDEpLCB0YXJnZXRzLmZsYXR0ZW4oKSkKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIDEuMCkKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgICAgICBsb3NzZXMuYXBwZW5kKGZsb2F0KGxvc3MuaXRlbSgpKSkKICAgICAgICB2YWxpZGF0aW9uX2YxLCBfLCBfID0gZXZhbHVhdGUobW9kZWwsIGxvYWRlcnNbMV0sIGRldmljZSkKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJlcG9jaD17ZXBvY2g6MDNkfSBsb3NzPXtucC5tZWFuKGxvc3Nlcyk6LjRmfSAiCiAgICAgICAgICAgIGYidmFsaWRhdGlvbl9tYWNyb19mMT17dmFsaWRhdGlvbl9mMTouNGZ9IgogICAgICAgICkKICAgICAgICBpZiB2YWxpZGF0aW9uX2YxID4gYmVzdDoKICAgICAgICAgICAgYmVzdCwgc3RhbGUgPSB2YWxpZGF0aW9uX2YxLCAwCiAgICAgICAgICAgIHRvcmNoLnNhdmUobW9kZWwuc3RhdGVfZGljdCgpLCBjaGVja3BvaW50KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHN0YWxlICs9IDEKICAgICAgICAgICAgaWYgc3RhbGUgPj0gYXJncy5wYXRpZW5jZToKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQoY2hlY2twb2ludCwgbWFwX2xvY2F0aW9uPWRldmljZSkpCiAgICB0ZXN0X2YxLCBhY3R1YWwsIHByZWRpY3RlZCA9IGV2YWx1YXRlKG1vZGVsLCBsb2FkZXJzWzJdLCBkZXZpY2UpCiAgICByZXBvcnQgPSBjbGFzc2lmaWNhdGlvbl9yZXBvcnQoCiAgICAgICAgYWN0dWFsLAogICAgICAgIHByZWRpY3RlZCwKICAgICAgICBsYWJlbHM9bGlzdChyYW5nZSgxLCBsZW4obGFiZWxfbmFtZXMpKSksCiAgICAgICAgdGFyZ2V0X25hbWVzPWxhYmVsX25hbWVzWzE6XSwKICAgICAgICB6ZXJvX2RpdmlzaW9uPTAsCiAgICAgICAgb3V0cHV0X2RpY3Q9VHJ1ZSwKICAgICkKICAgIG1vZGVsID0gbW9kZWwuY3B1KCkuZXZhbCgpCiAgICBzZXF1ZW5jZV9sZW5ndGggPSBpbnQocmF3WyJzZXF1ZW5jZV9sZW5ndGgiXSkKICAgIHRvcmNoLm9ubnguZXhwb3J0KAogICAgICAgIG1vZGVsLAogICAgICAgICgKICAgICAgICAgICAgdG9yY2guemVyb3MoMSwgc2VxdWVuY2VfbGVuZ3RoLCAzMywgNCksCiAgICAgICAgICAgIHRvcmNoLnplcm9zKDEsIGR0eXBlPXRvcmNoLmludDY0KSwKICAgICAgICApLAogICAgICAgIGFyZ3Mub3V0cHV0X2RpciAvICJtYXJ0aWFsX2FydHNfdGVtcG9yYWwub25ueCIsCiAgICAgICAgaW5wdXRfbmFtZXM9WyJsYW5kbWFya3MiLCAidGVjaG5pcXVlX2lkIl0sCiAgICAgICAgb3V0cHV0X25hbWVzPVsicGhhc2VfbG9naXRzIl0sCiAgICAgICAgZHluYW1pY19heGVzPXsKICAgICAgICAgICAgImxhbmRtYXJrcyI6IHswOiAiYmF0Y2giLCAxOiAidGltZSJ9LAogICAgICAgICAgICAidGVjaG5pcXVlX2lkIjogezA6ICJiYXRjaCJ9LAogICAgICAgICAgICAicGhhc2VfbG9naXRzIjogezA6ICJiYXRjaCIsIDE6ICJ0aW1lIn0sCiAgICAgICAgfSwKICAgICAgICBvcHNldF92ZXJzaW9uPTE4LAogICAgKQogICAgbWFwcGluZ3MgPSBqc29uLmxvYWRzKHN0cihyYXdbInBoYXNlX21hcHBpbmdzX2pzb24iXSkpCiAgICBtZXRhZGF0YSA9IHsKICAgICAgICAic2NoZW1hX3ZlcnNpb24iOiAiMi4wIiwKICAgICAgICAibW9kZWxfdHlwZSI6ICJ1bml2ZXJzYWwtdGVtcG9yYWwtcGhhc2UiLAogICAgICAgICJtb2RlbF92ZXJzaW9uIjogInVuaXZlcnNhbC10ZW1wb3JhbC12MSIsCiAgICAgICAgInJ1bnRpbWVfbW9kZSI6ICJwcmltYXJ5IiwKICAgICAgICAiYXJjaGl0ZWN0dXJlIjogInRlY2huaXF1ZS1jb25kaXRpb25lZC1zdGdjbi10Y24iLAogICAgICAgICJpbnB1dHMiOiB7CiAgICAgICAgICAgICJsYW5kbWFya3MiOiB7CiAgICAgICAgICAgICAgICAibmFtZSI6ICJsYW5kbWFya3MiLAogICAgICAgICAgICAgICAgImxheW91dCI6ICJCVFZDIiwKICAgICAgICAgICAgICAgICJqb2ludHMiOiAzMywKICAgICAgICAgICAgICAgICJjaGFubmVscyI6IFsieCIsICJ5IiwgInoiLCAidmlzaWJpbGl0eSJdLAogICAgICAgICAgICAgICAgInNlcXVlbmNlX2xlbmd0aCI6IHNlcXVlbmNlX2xlbmd0aCwKICAgICAgICAgICAgICAgICJub3JtYWxpemF0aW9uIjogImhpcC1jZW50ZXJlZC10b3Jzby1zY2FsZSIsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJ0ZWNobmlxdWUiOiB7CiAgICAgICAgICAgICAgICAibmFtZSI6ICJ0ZWNobmlxdWVfaWQiLAogICAgICAgICAgICAgICAgImR0eXBlIjogImludDY0IiwKICAgICAgICAgICAgICAgICJsYWJlbHMiOiB0ZWNobmlxdWVfbmFtZXMsCiAgICAgICAgICAgIH0sCiAgICAgICAgfSwKICAgICAgICAib3V0cHV0IjogewogICAgICAgICAgICAibmFtZSI6ICJwaGFzZV9sb2dpdHMiLAogICAgICAgICAgICAibGF5b3V0IjogIkJUQyIsCiAgICAgICAgICAgICJsYWJlbHMiOiBsYWJlbF9uYW1lcywKICAgICAgICB9LAogICAgICAgICJ0ZWNobmlxdWVzIjogbWFwcGluZ3MsCiAgICAgICAgInZhbGlkYXRpb24iOiB7CiAgICAgICAgICAgICJzcGxpdF91bml0IjogInNlc3Npb24iLAogICAgICAgICAgICAidmFsaWRhdGlvbl9tYWNyb19mMSI6IGJlc3QsCiAgICAgICAgICAgICJ0ZXN0X21hY3JvX2YxIjogdGVzdF9mMSwKICAgICAgICB9LAogICAgfQogICAgKGFyZ3Mub3V0cHV0X2RpciAvICJtYXJ0aWFsX2FydHNfdGVtcG9yYWwubWV0YWRhdGEuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhtZXRhZGF0YSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICAoYXJncy5vdXRwdXRfZGlyIC8gInRlc3RfcmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHByaW50KGYidGVzdF9tYWNyb19mMT17dGVzdF9mMTouNGZ9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==","universal-labels.json":"ewogICJzY2hlbWFfdmVyc2lvbiI6ICIxLjAiLAogICJwaGFzZXMiOiBbCiAgICAiX19QQURfXyIsCiAgICAiX19VTktOT1dOX18iLAogICAgIl9fVFJBQ0tJTkdfTE9TVF9fIiwKICAgICJQUkVQQVJBVElPTiIsCiAgICAiRU5UUlkiLAogICAgIkVYRUNVVElPTiIsCiAgICAiUEVBSyIsCiAgICAiUkVUUkFDVElPTiIsCiAgICAiUkVDT1ZFUlkiCiAgXSwKICAidGVjaG5pcXVlcyI6IHsKICAgICJqYWIiOiB7CiAgICAgICJuYXRpdmVfdG9fcGhhc2UiOiB7CiAgICAgICAgIkdVQVJEIjogIlBSRVBBUkFUSU9OIiwKICAgICAgICAiRVhURU5TSU9OIjogIkVYRUNVVElPTiIsCiAgICAgICAgIkZVTExfRVhURU5TSU9OIjogIlBFQUsiLAogICAgICAgICJSRVRSQUNUSU9OIjogIlJFVFJBQ1RJT04iLAogICAgICAgICJSRUNPVkVSWSI6ICJSRUNPVkVSWSIKICAgICAgfSwKICAgICAgInBoYXNlX3RvX25hdGl2ZSI6IHsKICAgICAgICAiUFJFUEFSQVRJT04iOiAiR1VBUkQiLAogICAgICAgICJFTlRSWSI6ICJFWFRFTlNJT04iLAogICAgICAgICJFWEVDVVRJT04iOiAiRVhURU5TSU9OIiwKICAgICAgICAiUEVBSyI6ICJGVUxMX0VYVEVOU0lPTiIsCiAgICAgICAgIlJFVFJBQ1RJT04iOiAiUkVUUkFDVElPTiIsCiAgICAgICAgIlJFQ09WRVJZIjogIlJFQ09WRVJZIgogICAgICB9CiAgICB9LAogICAgImZyb250LWtpY2siOiB7CiAgICAgICJuYXRpdmVfdG9fcGhhc2UiOiB7CiAgICAgICAgIlNUQU5DRSI6ICJQUkVQQVJBVElPTiIsCiAgICAgICAgIkNIQU1CRVIiOiAiRU5UUlkiLAogICAgICAgICJFWFRFTlNJT04iOiAiUEVBSyIsCiAgICAgICAgIlJFQ09JTCI6ICJSRVRSQUNUSU9OIiwKICAgICAgICAiUkVDT1ZFUlkiOiAiUkVDT1ZFUlkiCiAgICAgIH0sCiAgICAgICJwaGFzZV90b19uYXRpdmUiOiB7CiAgICAgICAgIlBSRVBBUkFUSU9OIjogIlNUQU5DRSIsCiAgICAgICAgIkVOVFJZIjogIkNIQU1CRVIiLAogICAgICAgICJFWEVDVVRJT04iOiAiRVhURU5TSU9OIiwKICAgICAgICAiUEVBSyI6ICJFWFRFTlNJT04iLAogICAgICAgICJSRVRSQUNUSU9OIjogIlJFQ09JTCIsCiAgICAgICAgIlJFQ09WRVJZIjogIlJFQ09WRVJZIgogICAgICB9CiAgICB9CiAgfQp9Cg=="}''')
for name, payload in encoded_files.items():
    (PIPELINE_DIR / name).write_bytes(base64.b64decode(payload))
print("Restored embedded files:")
for path in sorted(PIPELINE_DIR.iterdir()):
    print(" -", path.name, path.stat().st_size, "bytes")

## 4. Validate the raw bundle

The audit records dataset origin, session count, technique coverage,
annotation status, frames and phase support before training starts.


In [ ]:
payload = json.loads(DATA_FILE.read_text(encoding="utf-8"))
documents = payload.get("sessions") if isinstance(payload, dict) else payload
if not isinstance(documents, list) or not documents:
    raise ValueError("The JSON bundle contains no sessions.")

audit_rows = []
for document in documents:
    annotation = document.get("manual_annotation") or {}
    provenance = document.get("provenance") or {}
    audit_rows.append({
        "session_id": str(document.get("session_id", "")),
        "participant_id": str(
            document.get("participant_id")
            or (document.get("metadata") or {}).get("participantId")
            or ""
        ),
        "technique": str(document.get("technique_id", "")).lower(),
        "origin": str(provenance.get("origin", "real")),
        "annotation_status": str(annotation.get("status", "unverified")),
        "frames": len(document.get("frames") or []),
        "segments": len(annotation.get("segments") or []),
    })
raw_audit = pd.DataFrame(audit_rows)
display(raw_audit.groupby(
    ["technique", "origin", "annotation_status"], dropna=False
).agg(sessions=("session_id", "count"), frames=("frames", "sum")))
if raw_audit.session_id.duplicated().any():
    raise ValueError("Session IDs must be unique.")
if raw_audit.technique.nunique() < 2:
    raise ValueError("This technique-conditioned experiment requires two techniques.")

SYNTHETIC_ONLY = bool((raw_audit.origin == "synthetic").all())
EVALUATION_ORIGIN = (
    "synthetic_bootstrap_pipeline_check"
    if SYNTHETIC_ONLY else
    "contains_human_or_mixed_sessions_review_before_reporting"
)
print("Evaluation origin:", EVALUATION_ORIGIN)
if SYNTHETIC_ONLY:
    print("WARNING: resulting scores are not real-world model accuracy.")
raw_audit.to_csv(RUN_DIR / "raw_dataset_audit.csv", index=False)


## 5. Create the 90-frame MediaPipe-33 dataset

Native technique states are mapped to the shared phase vocabulary.
Pose coordinates are hip-centred and torso-scaled. Synthetic sessions
are included only because this run explicitly requests the bootstrap
dataset.


In [ ]:
import subprocess

TAPE_DIR = WORK_DIR / "tapes"
if TAPE_DIR.exists():
    shutil.rmtree(TAPE_DIR)
TAPE_DIR.mkdir(parents=True)
shutil.copy2(DATA_FILE, TAPE_DIR / DATA_FILE.name)
DATASET_PATH = WORK_DIR / "universal_temporal_dataset.npz"
subprocess.run([
    sys.executable, str(PIPELINE_DIR / "prepare_universal_dataset.py"),
    "--input-dir", str(TAPE_DIR),
    "--output", str(DATASET_PATH),
    "--label-config", str(PIPELINE_DIR / "universal-labels.json"),
    "--sequence-length", str(CFG.window),
    "--stride", str(CFG.stride),
    "--include-synthetic",
], check=True)

raw = np.load(DATASET_PATH, allow_pickle=False)
data = {
    "features": raw["features"].astype(np.float32),
    "technique_ids": raw["technique_ids"].astype(np.int64),
    "labels": raw["labels"].astype(np.int64),
    "mask": raw["mask"].astype(bool),
}
groups = raw["groups"].astype(str)
origins = raw["origins"].astype(str)
label_names = raw["label_names"].astype(str).tolist()
technique_names = raw["technique_names"].astype(str).tolist()
phase_mappings = json.loads(str(raw["phase_mappings_json"]))
LABEL_TO_ID = {name: index for index, name in enumerate(label_names)}
TECHNIQUE_TO_ID = {name: index for index, name in enumerate(technique_names)}

print("Features:", data["features"].shape)
print("Labels:", data["labels"].shape)
print("Sessions:", len(np.unique(groups)))
print("Techniques:", technique_names)
print("Phases:", label_names)


## 6. Fixed session-separated split

Sessions are separated by technique into training, validation and
test groups. Every overlapping window from one session remains in the
same split. The same split is reused for all three training seeds so
their variation measures initialization/training variation rather
than different test samples.

Synthetic sessions do not contain real participant identities, so
this run uses session-level grouping. A final human study should use
participant-level separation or leave-one-participant-out evaluation.


In [ ]:
def fixed_stratified_group_split(groups, technique_ids, cfg):
    rng = np.random.default_rng(cfg.split_seed)
    allocation = {"train": [], "validation": [], "test": []}
    for technique_id, technique_name in enumerate(technique_names):
        technique_groups = np.unique(groups[technique_ids == technique_id])
        rng.shuffle(technique_groups)
        count = len(technique_groups)
        if count < 6:
            raise ValueError(
                f"{technique_name} needs at least six independent sessions."
            )
        test_count = max(1, int(round(count * cfg.test_fraction)))
        validation_count = max(1, int(round(count * cfg.validation_fraction)))
        allocation["test"].extend(technique_groups[:test_count])
        allocation["validation"].extend(
            technique_groups[test_count:test_count + validation_count]
        )
        allocation["train"].extend(
            technique_groups[test_count + validation_count:]
        )
    indexes = {
        name: np.flatnonzero(np.isin(groups, selected))
        for name, selected in allocation.items()
    }
    sets = {name: set(values) for name, values in allocation.items()}
    assert sets["train"].isdisjoint(sets["validation"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["validation"].isdisjoint(sets["test"])
    return allocation, indexes

split_groups, split_ids = fixed_stratified_group_split(
    groups, data["technique_ids"], CFG
)
split_rows = []
for split_name, selected_groups in split_groups.items():
    ids = split_ids[split_name]
    print(
        split_name,
        "sessions=", len(selected_groups),
        "windows=", len(ids),
        "techniques=", sorted(set(data["technique_ids"][ids].tolist())),
    )
    split_rows.extend(
        {"split": split_name, "group": group}
        for group in selected_groups
    )
pd.DataFrame(split_rows).to_csv(RUN_DIR / "split_groups.csv", index=False)


## 7. Recover complete session timelines

Complete timelines allow unique-frame evaluation rather than counting
the same frame repeatedly in overlapping windows. They are also used
for transition-boundary and completed-sequence measurements.


In [ ]:
sys.path.insert(0, str(PIPELINE_DIR))
from prepare_dataset import (
    normalize_pose, select_landmarks, verified_manual_labels
)
from train_universal_model import UniversalTemporalSTGCN

def recover_sessions(documents):
    recovered = []
    for document in documents:
        technique = str(document.get("technique_id", "")).lower()
        if technique not in TECHNIQUE_TO_ID:
            continue
        frames = document.get("frames") or []
        native = verified_manual_labels(document, len(frames), True)
        if native is None:
            continue
        mapping = phase_mappings[technique]["native_to_phase"]
        universal = [
            state if state in {"__UNKNOWN__", "__TRACKING_LOST__"}
            else mapping[state]
            for state in native
        ]
        annotation = document.get("manual_annotation") or {}
        rep_values = [
            int(segment.get("rep", 0) or 0)
            for segment in annotation.get("segments") or []
        ]
        session_id = str(document.get("session_id"))
        recovered.append({
            "group": f"{technique}:{session_id}",
            "session_id": session_id,
            "technique": technique,
            "technique_id": TECHNIQUE_TO_ID[technique],
            "fps": float(document.get("nominal_fps", 30)),
            "x": np.asarray([
                normalize_pose(select_landmarks(frame)) for frame in frames
            ], dtype=np.float32),
            "y": np.asarray([LABEL_TO_ID[state] for state in universal]),
            "true_repetitions": max(rep_values, default=0),
        })
    return recovered

complete_sessions = recover_sessions(documents)
by_group = {session["group"]: session for session in complete_sessions}
if len(by_group) != len(np.unique(groups)):
    raise ValueError("Could not recover every prepared session timeline.")
split_sessions = {
    name: [by_group[group] for group in selected]
    for name, selected in split_groups.items()
}
print({name: len(values) for name, values in split_sessions.items()})


## 8. Model, loaders and training

The architecture matches the deployed technique-conditioned
ST-GCN/TCN:

`MediaPipe-33 graph blocks → visibility-weighted pooling → technique embedding → temporal convolutions → per-frame phase logits`

Class weights are calculated from training frames only. Validation
macro-F1 chooses each seed's checkpoint. Test labels do not influence
training or checkpoint selection.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_loader(ids, shuffle, seed=0):
    dataset = TensorDataset(
        torch.from_numpy(data["features"][ids]).float(),
        torch.from_numpy(data["technique_ids"][ids]).long(),
        torch.from_numpy(data["labels"][ids]).long(),
        torch.from_numpy(data["mask"][ids]).bool(),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=CFG.batch_size, shuffle=shuffle,
        generator=generator if shuffle else None
    )

ACTIVE_CLASS_IDS = sorted(
    set(data["labels"][data["mask"]].tolist()) - {LABEL_TO_ID["__PAD__"]}
)

@torch.no_grad()
def window_predictions(model, loader):
    model.eval()
    true_values, predicted_values = [], []
    for x, technique_ids, labels, mask in loader:
        logits = model(x.to(DEVICE), technique_ids.to(DEVICE))
        valid = mask.to(DEVICE) & labels.to(DEVICE).ne(LABEL_TO_ID["__PAD__"])
        true_values.append(labels.to(DEVICE)[valid].cpu().numpy())
        predicted_values.append(logits.argmax(-1)[valid].cpu().numpy())
    return np.concatenate(true_values), np.concatenate(predicted_values)

def macro_f1(y_true, y_pred):
    return float(f1_score(
        y_true, y_pred, labels=ACTIVE_CLASS_IDS,
        average="macro", zero_division=0
    ))

def training_weights(train_ids):
    values = data["labels"][train_ids][data["mask"][train_ids]]
    counts = np.bincount(values, minlength=len(label_names)).astype(float)
    weights = counts.sum() / np.maximum(counts, 1)
    weights /= max(weights[ACTIVE_CLASS_IDS].mean(), 1e-8)
    weights[LABEL_TO_ID["__PAD__"]] = 0
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_one_seed(seed):
    seed_everything(seed)
    train_loader = make_loader(split_ids["train"], True, seed)
    validation_loader = make_loader(split_ids["validation"], False)
    model = UniversalTemporalSTGCN(
        len(label_names), len(technique_names),
        CFG.hidden_size, CFG.dropout
    ).to(DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=training_weights(split_ids["train"]),
        ignore_index=LABEL_TO_ID["__PAD__"],
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.learning_rate,
        weight_decay=CFG.weight_decay
    )
    best_score, best_state, stale, history = -1.0, None, 0, []
    for epoch in range(1, CFG.epochs + 1):
        model.train()
        losses = []
        for x, technique_ids, labels, mask in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(x.to(DEVICE), technique_ids.to(DEVICE))
            targets = labels.to(DEVICE).masked_fill(~mask.to(DEVICE), 0)
            loss = criterion(
                logits.flatten(0, 1), targets.flatten()
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(float(loss.item()))
        actual, predicted = window_predictions(model, validation_loader)
        score = macro_f1(actual, predicted)
        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "validation_macro_f1": score,
        })
        print(
            f"seed={seed} epoch={epoch:03d} "
            f"loss={np.mean(losses):.5f} val_macro_f1={score:.5f}"
        )
        if score > best_score:
            best_score, stale = score, 0
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            stale += 1
        if stale >= CFG.patience:
            print("Early stopping")
            break
    model.load_state_dict(best_state)
    return model.cpu().eval(), best_score, pd.DataFrame(history)


## 9. Unique-frame session evaluation

Overlapping window probabilities are averaged to reconstruct every
held-out session. Reported test frame metrics therefore count each
original session frame once.


In [ ]:
def window_starts(length, window, stride):
    starts = list(range(0, max(1, length - window + 1), stride))
    last = max(0, length - window)
    if last not in starts:
        starts.append(last)
    return starts

@torch.no_grad()
def predict_session(model, session, corruption=None, seed=0):
    model.eval()
    x = session["x"].copy()
    if corruption is not None:
        x = corruption(x, np.random.default_rng(seed))
    starts = window_starts(len(x), CFG.window, CFG.stride)
    windows, valid_counts = [], []
    for start in starts:
        valid = min(CFG.window, len(x) - start)
        window = np.zeros((CFG.window, 33, 4), dtype=np.float32)
        window[:valid] = x[start:start + valid]
        windows.append(window)
        valid_counts.append(valid)
    batch = torch.from_numpy(np.stack(windows)).float().to(DEVICE)
    technique = torch.full(
        (len(windows),), session["technique_id"],
        dtype=torch.long, device=DEVICE
    )
    probabilities = torch.softmax(model(batch, technique), -1).cpu().numpy()
    total = np.zeros((len(x), len(label_names)), dtype=np.float64)
    counts = np.zeros(len(x), dtype=np.float64)
    for start, valid, window_probabilities in zip(
        starts, valid_counts, probabilities
    ):
        total[start:start + valid] += window_probabilities[:valid]
        counts[start:start + valid] += 1
    if np.any(counts == 0):
        raise RuntimeError("A session frame received no prediction.")
    mean_probabilities = total / counts[:, None]
    return mean_probabilities.argmax(-1), mean_probabilities

def predict_sessions(model, sessions, corruption=None, seed=0):
    rows, actual, predicted = [], [], []
    session_outputs = []
    for index, session in enumerate(sessions):
        y_pred, probabilities = predict_session(
            model, session, corruption, seed + index
        )
        actual.append(session["y"])
        predicted.append(y_pred)
        session_outputs.append((session, y_pred, probabilities))
        for frame, (true_id, predicted_id) in enumerate(
            zip(session["y"], y_pred)
        ):
            rows.append({
                "session_id": session["session_id"],
                "technique": session["technique"],
                "frame": frame,
                "true_phase": label_names[int(true_id)],
                "predicted_phase": label_names[int(predicted_id)],
                "confidence": float(probabilities[frame, predicted_id]),
            })
    return (
        np.concatenate(actual), np.concatenate(predicted),
        session_outputs, pd.DataFrame(rows)
    )

def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(
            y_true, y_pred, labels=ACTIVE_CLASS_IDS,
            average="macro", zero_division=0
        )),
        "macro_recall": float(recall_score(
            y_true, y_pred, labels=ACTIVE_CLASS_IDS,
            average="macro", zero_division=0
        )),
        "macro_f1": macro_f1(y_true, y_pred),
        "weighted_f1": float(f1_score(
            y_true, y_pred, labels=ACTIVE_CLASS_IDS,
            average="weighted", zero_division=0
        )),
    }


In [ ]:
trained_models, validation_scores, run_rows = {}, {}, []
for seed in CFG.seeds:
    model, validation_f1, history = train_one_seed(seed)
    trained_models[seed] = model
    validation_scores[seed] = validation_f1
    history.to_csv(RUN_DIR / f"training_history_seed_{seed}.csv", index=False)
    torch.save(model.state_dict(), RUN_DIR / f"checkpoint_seed_{seed}.pt")
    model = model.to(DEVICE)
    y_true, y_pred, _, _ = predict_sessions(
        model, split_sessions["test"], seed=seed
    )
    metrics = classification_metrics(y_true, y_pred)
    metrics.update({
        "seed": seed,
        "method": "technique_conditioned_stgcn_tcn",
        "validation_macro_f1": validation_f1,
    })
    run_rows.append(metrics)

# Technique-conditioned training-frame majority baseline.
majority_by_technique = {}
for technique in technique_names:
    labels = np.concatenate([
        session["y"] for session in split_sessions["train"]
        if session["technique"] == technique
    ])
    majority_by_technique[technique] = int(
        np.bincount(labels, minlength=len(label_names)).argmax()
    )
baseline_true = np.concatenate([
    session["y"] for session in split_sessions["test"]
])
baseline_pred = np.concatenate([
    np.full(len(session["y"]), majority_by_technique[session["technique"]])
    for session in split_sessions["test"]
])
baseline_metrics = classification_metrics(baseline_true, baseline_pred)
baseline_metrics.update({
    "seed": np.nan, "method": "technique_majority_baseline",
    "validation_macro_f1": np.nan,
})
run_rows.append(baseline_metrics)

results = pd.DataFrame(run_rows)
display(results)
results.to_csv(RUN_DIR / "metrics_by_run.csv", index=False)
learned = results[results.method == "technique_conditioned_stgcn_tcn"]
summary = learned.select_dtypes("number").drop(
    columns=["seed"], errors="ignore"
).agg(["mean", "std"])
display(summary)
summary.to_csv(RUN_DIR / "metrics_summary.csv")


## 10. Best validation-selected model: phase and technique analysis

The deployment candidate is selected using validation macro-F1 only,
never test performance.


In [ ]:
best_seed = max(validation_scores, key=validation_scores.get)
final_model = trained_models[best_seed].to(DEVICE)
y_true, y_pred, session_outputs, prediction_rows = predict_sessions(
    final_model, split_sessions["test"], seed=best_seed
)
prediction_rows.to_csv(RUN_DIR / "test_frame_predictions.csv", index=False)

report = classification_report(
    y_true, y_pred, labels=ACTIVE_CLASS_IDS,
    target_names=[label_names[index] for index in ACTIVE_CLASS_IDS],
    output_dict=True, zero_division=0
)
with open(RUN_DIR / "classification_report.json", "w") as handle:
    json.dump(report, handle, indent=2)
display(pd.DataFrame(report).T)

technique_rows = []
for technique in technique_names:
    selected = prediction_rows.technique.eq(technique).to_numpy()
    technique_true = np.asarray([
        LABEL_TO_ID[value] for value in prediction_rows.loc[selected, "true_phase"]
    ])
    technique_pred = np.asarray([
        LABEL_TO_ID[value] for value in prediction_rows.loc[selected, "predicted_phase"]
    ])
    row = classification_metrics(technique_true, technique_pred)
    row["technique"] = technique
    technique_rows.append(row)
technique_table = pd.DataFrame(technique_rows)
display(technique_table)
technique_table.to_csv(RUN_DIR / "metrics_by_technique.csv", index=False)

matrix = confusion_matrix(y_true, y_pred, labels=ACTIVE_CLASS_IDS)
active_names = [label_names[index] for index in ACTIVE_CLASS_IDS]
plt.figure(figsize=(10, 8))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues",
    xticklabels=active_names, yticklabels=active_names
)
plt.xlabel("Predicted phase")
plt.ylabel("True phase")
plt.title("Held-out session phase confusion matrix")
plt.tight_layout()
plt.savefig(RUN_DIR / "confusion_matrix.png", dpi=200)
plt.show()


## 11. Boundary and completed-sequence evaluation

A five-frame majority filter suppresses isolated label flicker before
transition analysis. A predicted boundary matches a true boundary
only if the destination phase is the same and its timing is within
the declared tolerance.

Completed-sequence counts use technique-specific canonical orders.
This is a coarse offline diagnostic; the authoritative runtime ordered
decoder must still be evaluated separately in the application.


In [ ]:
def majority_smooth(values, width):
    radius = width // 2
    output = values.copy()
    for index in range(len(values)):
        local = values[max(0, index-radius):min(len(values), index+radius+1)]
        output[index] = np.bincount(
            local, minlength=len(label_names)
        ).argmax()
    return output

def transition_events(values):
    indexes = np.flatnonzero(values[1:] != values[:-1]) + 1
    return [(int(index), int(values[index])) for index in indexes]

def boundary_result(true_values, predicted_values, tolerance):
    true_events = transition_events(true_values)
    predicted_events = transition_events(predicted_values)
    used, errors = set(), []
    for true_index, destination in true_events:
        candidates = [
            (abs(pred_index - true_index), candidate_index)
            for candidate_index, (pred_index, pred_destination)
            in enumerate(predicted_events)
            if candidate_index not in used
            and pred_destination == destination
            and abs(pred_index - true_index) <= tolerance
        ]
        if candidates:
            error, candidate = min(candidates)
            used.add(candidate)
            errors.append(error)
    matches = len(errors)
    return {
        "matches": matches,
        "true_boundaries": len(true_events),
        "predicted_boundaries": len(predicted_events),
        "boundary_precision": matches / max(len(predicted_events), 1),
        "boundary_recall": matches / max(len(true_events), 1),
        "boundary_f1": (
            2 * matches / max(len(true_events) + len(predicted_events), 1)
        ),
        "boundary_mae_frames": float(np.mean(errors)) if errors else np.nan,
    }

PHASE_ORDERS = {
    "jab": ["PREPARATION", "EXECUTION", "PEAK", "RETRACTION", "RECOVERY"],
    "front-kick": ["PREPARATION", "ENTRY", "PEAK", "RETRACTION", "RECOVERY"],
}

def completed_sequences(values, technique):
    collapsed = [
        value for index, value in enumerate(values)
        if index == 0 or value != values[index - 1]
    ]
    ignored = {
        LABEL_TO_ID["__PAD__"], LABEL_TO_ID["__UNKNOWN__"],
        LABEL_TO_ID["__TRACKING_LOST__"]
    }
    phases = [label_names[value] for value in collapsed if value not in ignored]
    order = PHASE_ORDERS[technique]
    position, count = 0, 0
    for phase in phases:
        if phase == order[position]:
            position += 1
            if position == len(order):
                count += 1
                position = 0
        elif phase == order[0]:
            position = 1
    return count

boundary_rows, repetition_rows = [], []
for session, predicted, _ in session_outputs:
    smoothed = majority_smooth(
        predicted, CFG.boundary_smoothing_frames
    )
    boundary = boundary_result(
        session["y"], smoothed, CFG.boundary_tolerance_frames
    )
    boundary.update({
        "session_id": session["session_id"],
        "technique": session["technique"],
    })
    boundary_rows.append(boundary)
    repetition_rows.append({
        "session_id": session["session_id"],
        "technique": session["technique"],
        "true_repetitions": session["true_repetitions"],
        "predicted_repetitions": completed_sequences(
            smoothed, session["technique"]
        ),
    })

boundary_table = pd.DataFrame(boundary_rows)
repetition_table = pd.DataFrame(repetition_rows)
display(boundary_table)
display(repetition_table)
boundary_table.to_csv(RUN_DIR / "boundary_metrics_by_session.csv", index=False)
repetition_table.to_csv(RUN_DIR / "repetition_counts_by_session.csv", index=False)

total_true = int(repetition_table.true_repetitions.sum())
total_predicted = int(repetition_table.predicted_repetitions.sum())
matched = int(np.minimum(
    repetition_table.true_repetitions,
    repetition_table.predicted_repetitions
).sum())
sequence_summary = {
    "boundary_precision_mean": float(boundary_table.boundary_precision.mean()),
    "boundary_recall_mean": float(boundary_table.boundary_recall.mean()),
    "boundary_f1_mean": float(boundary_table.boundary_f1.mean()),
    "boundary_mae_frames_mean": float(boundary_table.boundary_mae_frames.mean()),
    "repetition_count_mae": float(np.mean(np.abs(
        repetition_table.true_repetitions
        - repetition_table.predicted_repetitions
    ))),
    "coarse_repetition_precision": matched / max(total_predicted, 1),
    "coarse_repetition_recall": matched / max(total_true, 1),
    "false_repetitions_per_minute_on_unrelated_motion": None,
    "false_repetition_note": (
        "Not measurable: this bootstrap contains no dedicated "
        "unrelated-motion negative sessions."
    ),
}
print(json.dumps(sequence_summary, indent=2))
with open(RUN_DIR / "sequence_summary.json", "w") as handle:
    json.dump(sequence_summary, handle, indent=2)


## 12. Robustness

These controlled corruptions test sensitivity to normalized-coordinate
noise and missing landmarks. They do not reproduce every real
MediaPipe failure mode.


In [ ]:
def coordinate_noise(std):
    def apply(values, rng):
        output = values.copy()
        output[..., :3] += rng.normal(
            0, std, output[..., :3].shape
        ).astype(np.float32)
        return output
    return apply

def missing_landmarks(probability):
    def apply(values, rng):
        output = values.copy()
        missing = rng.random(output.shape[:2]) < probability
        output[missing, :3] = 0
        output[missing, 3] = 0
        return output
    return apply

robustness_conditions = [
    ("clean", None),
    ("noise_0.005", coordinate_noise(0.005)),
    ("noise_0.010", coordinate_noise(0.010)),
    ("missing_0.05", missing_landmarks(0.05)),
    ("missing_0.10", missing_landmarks(0.10)),
]
robustness_rows = []
for name, corruption in robustness_conditions:
    robust_true, robust_pred, _, _ = predict_sessions(
        final_model, split_sessions["test"],
        corruption=corruption, seed=7300
    )
    row = classification_metrics(robust_true, robust_pred)
    row["condition"] = name
    robustness_rows.append(row)
robustness = pd.DataFrame(robustness_rows)
display(robustness)
robustness.to_csv(RUN_DIR / "robustness.csv", index=False)


## 13. ONNX parity and model-only CPU latency

The exact exported ONNX file should be used in the application.
Browser and complete camera-to-feedback latency remain separate
system-level measurements.


In [ ]:
import onnx
import onnxruntime as ort

final_model = final_model.cpu().eval()
onnx_path = RUN_DIR / "martial_arts_temporal.onnx"
example_landmarks = torch.from_numpy(
    data["features"][split_ids["test"][:1]]
).float()
example_technique = torch.from_numpy(
    data["technique_ids"][split_ids["test"][:1]]
).long()
torch.onnx.export(
    final_model,
    (example_landmarks, example_technique),
    onnx_path,
    input_names=["landmarks", "technique_id"],
    output_names=["phase_logits"],
    dynamic_axes={
        "landmarks": {0: "batch", 1: "time"},
        "technique_id": {0: "batch"},
        "phase_logits": {0: "batch", 1: "time"},
    },
    opset_version=18,
    dynamo=False,
)
onnx.checker.check_model(onnx.load(onnx_path))
runtime = ort.InferenceSession(
    str(onnx_path), providers=["CPUExecutionProvider"]
)
with torch.no_grad():
    torch_output = final_model(
        example_landmarks, example_technique
    ).numpy()
onnx_output = runtime.run(None, {
    "landmarks": example_landmarks.numpy(),
    "technique_id": example_technique.numpy(),
})[0]

for _ in range(10):
    runtime.run(None, {
        "landmarks": example_landmarks.numpy(),
        "technique_id": example_technique.numpy(),
    })
latencies = []
for _ in range(200):
    started = time.perf_counter()
    runtime.run(None, {
        "landmarks": example_landmarks.numpy(),
        "technique_id": example_technique.numpy(),
    })
    latencies.append((time.perf_counter() - started) * 1000)

parity = {
    "max_abs_error": float(np.max(np.abs(torch_output - onnx_output))),
    "mean_abs_error": float(np.mean(np.abs(torch_output - onnx_output))),
    "predicted_labels_equal": bool(np.array_equal(
        torch_output.argmax(-1), onnx_output.argmax(-1)
    )),
    "passed_at_1e-4": bool(
        np.max(np.abs(torch_output - onnx_output)) < 1e-4
        and np.array_equal(
            torch_output.argmax(-1), onnx_output.argmax(-1)
        )
    ),
    "cpu_latency_median_ms": float(np.median(latencies)),
    "cpu_latency_p95_ms": float(np.percentile(latencies, 95)),
    "onnx_sha256": sha256_file(onnx_path),
}
print(json.dumps(parity, indent=2))
with open(RUN_DIR / "onnx_parity_latency.json", "w") as handle:
    json.dump(parity, handle, indent=2)


## 14. Metadata, provenance and downloadable evidence

The archive explicitly labels the evaluation origin. When the default
bootstrap is used, high classification scores mean the implementation
learned generator-produced patterns; they do not establish human,
camera or martial-arts generalization.


In [ ]:
metadata = {
    "schema_version": "3.0",
    "model_type": "temporal-phase-classifier",
    "architecture": "technique-conditioned-stgcn-tcn",
    "model_version": "research-evaluation-v1",
    "run_id": RUN_ID,
    "inputs": {
        "landmarks": {
            "layout": "BTVC",
            "frames": CFG.window,
            "joints": 33,
            "channels": ["x", "y", "z", "visibility"],
            "normalization": "hip-centered-torso-scale",
        },
        "technique_id": {"labels": technique_names},
    },
    "output": {"layout": "BTC", "phase_labels": label_names},
    "techniques": phase_mappings,
    "selected_seed": best_seed,
    "selection_rule": "highest validation macro F1",
    "evaluation_origin": EVALUATION_ORIGIN,
}
with open(RUN_DIR / "model_metadata.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

provenance = {
    "run_id": RUN_ID,
    "data_source": CFG.data_source,
    "source_url": SAMPLE_URL if CFG.data_source == "github_sample" else None,
    "pinned_commit": PINNED_COMMIT if CFG.data_source == "github_sample" else None,
    "dataset_sha256": sha256_file(DATA_FILE),
    "dataset_origin": EVALUATION_ORIGIN,
    "configuration": asdict(CFG),
    "split_unit": "session",
    "split_groups": split_groups,
    "active_evaluation_classes": [
        label_names[index] for index in ACTIVE_CLASS_IDS
    ],
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "onnxruntime": ort.__version__,
    },
    "device": str(DEVICE),
    "limitations": [
        "Synthetic bootstrap scores are pipeline checks, not human accuracy.",
        "The synthetic split is session-level because it has no real participants.",
        "No unrelated-motion negative sessions are available for false-repetition rate.",
        "The offline repetition count is not the authoritative runtime ordered decoder.",
        "Browser and end-to-end system latency require separate measurement.",
    ],
}
with open(RUN_DIR / "provenance.json", "w") as handle:
    json.dump(provenance, handle, indent=2)

archive = shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
print("Results directory:", RUN_DIR)
print("Archive:", archive)
print("Archive bytes:", Path(archive).stat().st_size)

from google.colab import files
files.download(archive)


## Expected interpretation

A successful bootstrap run should:

- create valid train/validation/test session groups for both techniques;
- train all three seeds without leakage;
- beat the technique-majority baseline on macro-F1;
- produce a readable confusion matrix and per-phase report;
- pass PyTorch/ONNX parity at `1e-4`;
- show model-only latency suitable for later real-time testing; and
- download one timestamped ZIP containing the evidence.

There is no guaranteed expected accuracy value. The measured values
must come from the executed notebook. Even a very high synthetic score
is reported only as **synthetic bootstrap pipeline performance**.

Final thesis claims about real phase-classification accuracy require
human-verified recordings, participant-separated testing, unrelated
movement negatives and evaluation of the deployed ordered decoder.
